# load dependecies

In [ ]:
## Initialzing and loading required libraries and subfunctions
import numpy as np
import matplotlib.pyplot as plt
import copy
import yasa
from mne.filter import resample
import pynapple as nap
import seaborn as sns
import pandas as pd
from sklearn.preprocessing import normalize
import requests
from io import BytesIO
import sails
from ssqueezepy import cwt
import re
from scipy.stats import entropy
import os, sys

import scipy
from scipy import signal
from scipy.interpolate import griddata
from scipy.signal import correlate
from scipy.stats import pearsonr
from scipy.fft import fft
from scipy.spatial.distance import euclidean
from scipy.signal import spectrogram
from scipy.io import loadmat
import scipy.fft
import scipy.stats
import scipy.io as sio
from scipy.signal import hilbert

import emd as emd
import emd.sift as sift
import emd.spectra as spectra

from neurodsp.sim import sim_combined
from neurodsp.plts import plot_time_series, plot_timefrequency
from neurodsp.utils import create_times
from neurodsp.timefrequency.wavelets import compute_wavelet_transform
from neurodsp.filt import filter_signal

# Load required libraries
import numpy as np
from scipy.io import loadmat
from scipy.signal import hilbert
from scipy.interpolate import interp1d
import matplotlib.pyplot as plt
import seaborn as sns
from neurodsp.filt import filter_signal, filter_signal_fir, design_fir_filter
import emd
import pandas as pd
from sklearn.preprocessing import Normalizer
from tqdm import tqdm
import plotly.express as px
import copy
import umap.umap_ as umap
from scipy.spatial import cKDTree
import pickle
from mpl_toolkits.axes_grid1 import make_axes_locatable
import matplotlib.colors as mcolors
import matplotlib.ticker as ticker
from matplotlib.cm import ScalarMappable

## UTILS

for rel in ('src', '../src'):
    p = os.path.abspath(os.path.join(os.getcwd(), rel))
    if os.path.isdir(p) and p not in sys.path:
        sys.path.insert(0, p)

from utils import *
from detect_pt import *

from scipy.io import loadmat
import numpy as np
from neurodsp.filt import filter_signal
import copy
import emd
from scipy.spatial import cKDTree
from tqdm import tqdm

sns.set(style='white', context='notebook')

In [ ]:
path_to_config = '/Users/amir/Desktop/for Abdel/emd_masksift_CA1_config.yml'
config = emd.sift.SiftConfig.from_yaml_file(path_to_config)

# preprocessing

In [ ]:
def extract_cycle_info(imfs, imf_frequencies):

  waveforms = pd.DataFrame()
  all_trials = pd.DataFrame()
  raw_wavelets = []
  all_FPPs = []

  theta_range = [5, 12]
  frequencies = np.arange(15, 141, 1)  # Logarithmically spaced frequencies from 15 to 300 Hz
  angles=np.linspace(-180,180,19)
  fs = 1000

  for idx, imf in enumerate(imfs):
    cycle_data = get_cycle_data(imf[:, 5], fs)

    amp_thresh = np.percentile(cycle_data['IA'], 25) # higher than 25th percentile of the data
    lo_freq_duration = fs/5  # restrict the analysis to 5-12 Hz
    hi_freq_duration = fs/12

    conditions = ['is_good==1',
                        f'duration_samples<{lo_freq_duration}',
                        f'duration_samples>{hi_freq_duration}',
                        f'max_amp>{amp_thresh}']
    print(len(cycle_data['theta_imf']))
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    
    # Check if any cycles satisfy the conditions
    if all_cycles is None or all_cycles.chain_vect.size == 0:
        print("No cycles satisfy the conditions.")
        return pd.DataFrame(), pd.DataFrame(), []
    
    subset_cycles_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_cycles_df['index'].values

    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycles_inds = arrange_cycle_inds(all_cycles_inds)

    freqs = imf_frequencies[idx]
    sub_theta, theta, supra_theta = tg_split(freqs, theta_range)
    supra_theta_sig = np.sum(imf.T[supra_theta], axis=0)

    # # Corrected Wavelet Transform Computation
    raw_data=sails.wavelet.morlet(supra_theta_sig, freqs=frequencies, sample_rate=fs, ncycles=5,ret_mode='power', normalise='simple')
    raw_wavelets.append(raw_data)
    supraPlot = scipy.stats.zscore(raw_data, axis=1)
    FPP = bin_tf_to_fpp(cycles_inds, supraPlot, bin_count=19)
    all_FPPs.append(FPP)

    # Compute mode frequency for each cycle
    mode_freqs, entropies = compute_mode_frequency_and_entropy(FPP, frequencies, angles)

    all_waveforms, _ = emd.cycles.phase_align(cycle_data['IP'], cycle_data['theta_imf'],
                                                            cycles=all_cycles.iterate(through='subset'), npoints=100)
    all_waveforms = pd.DataFrame(all_waveforms.T)

    waveforms = pd.concat([waveforms, all_waveforms])

    trial = all_cycles.get_metric_dataframe(subset=True)
    trial['mode_freqs'] = mode_freqs
    trial['entropy'] = entropies
    all_trials = pd.concat([all_trials, trial])

  return waveforms, all_trials, all_FPPs

## extract the data

In [ ]:
# =============================================================================
# ========== FPP-derived spectral signatures over the whole dataset ===========
# =============================================================================
import numpy as np
import pandas as pd
import os, re, warnings
import pywt
from scipy.stats import zscore
import emd

# ------------- Morlet amplitude CWT (same as preprocess v2 path) -------------
def _morlet_amp_spectrogram(x, fs, freqs_hz, w=6.0):
    """
    Morlet CWT amplitude (|complex CWT|), using pywt.cwt.
    Returns array [n_freq, n_time].

    We preserve the original scipy morlet2 scale mapping:
        scale = (w * fs) / (2π f)
    by setting the complex Morlet center frequency to w/(2π).
    """
    freqs_hz = np.asarray(freqs_hz, float)
    if np.any(freqs_hz <= 0):
        raise ValueError("freqs_hz must be strictly positive for CWT")

    center_freq = w / (2.0 * np.pi)
    wavelet = pywt.ContinuousWavelet(f"cmor1.5-{center_freq:.6f}")
    scales = (center_freq * fs) / freqs_hz

    cwt_mat, _ = pywt.cwt(
        data=np.asarray(x, float),
        scales=scales,
        wavelet=wavelet,
        sampling_period=1.0 / fs,
    )
    return np.abs(cwt_mat)

# -------------------------- Theta IMF picker (v2-ish) -------------------------
def _choose_theta_imf(imf, fs, imf_centers_hz, theta_band=(5.0, 12.0), prefer_idx=5):
    """
    Prefer a fixed theta IMF index if available; otherwise pick IMF with center
    frequency closest to the theta band center.
    """
    if imf.shape[1] > prefer_idx:
        return prefer_idx
    lo, hi = theta_band
    centers = np.asarray(imf_centers_hz).astype(float)
    if centers.ndim == 1 and centers.size == imf.shape[1]:
        band_center = 0.5*(lo + hi)
        return int(np.argmin(np.abs(centers - band_center)))
    warnings.warn("Unexpected imf_centers_hz shape; defaulting to last IMF.")
    return imf.shape[1] - 1

# --------------------------- Cycle bounds utility -----------------------------
def _cycle_bounds_from_inds(all_cycles_inds):
    """
    Convert per-cycle index arrays -> [start, end] bounds for each cycle.
    """
    bounds = []
    for cyc in all_cycles_inds:
        cyc = np.asarray(cyc)
        cyc = cyc[np.isfinite(cyc)]
        if cyc.size > 1:
            s, e = int(cyc[0]), int(cyc[-1])
            if e > s:
                bounds.append([s, e])
    return np.array(bounds, dtype=int)

# ---------------- FPP-derived spectral signatures for ONE segment -------------
def spectral_signatures_from_fpp_for_segment(
    imf, imf_centers_hz, fs,
    theta_band=(5,12),
    freq_vec=np.arange(15, 141, 1),
    morlet_w=6.0,
    n_bins=19,
    normalize='zscore',   # 'zscore' or 'none'
):
    """
    Returns:
      sigs_per_cycle : (n_cycles, n_freq) spectral signatures (FPP-based)
      n_cycles       : int, number of valid cycles used
    """
    # 1) theta IMF & cycle selection (same filters you use)
    theta_idx = _choose_theta_imf(imf, fs, imf_centers_hz, theta_band=theta_band, prefer_idx=5)
    cycle_data = get_cycle_data(imf[:, theta_idx], fs=fs)
    if cycle_data is None or 'IA' not in cycle_data:
        return np.zeros((0, len(freq_vec))), 0

    amp_thresh = np.percentile(cycle_data['IA'], 25)
    lo_dur = fs / float(theta_band[0])  # 5 Hz -> shortest period
    hi_dur = fs / float(theta_band[1])  # 12 Hz -> longest period
    conditions = [
        'is_good==1',
        f'duration_samples<{lo_dur}',
        f'duration_samples>{hi_dur}',
        f'max_amp>{amp_thresh}',
    ]
    all_cycles = get_cycles_with_conditions(cycle_data['cycles'], conditions)
    if all_cycles is None or getattr(all_cycles, 'chain_vect', np.array([])).size == 0:
        return np.zeros((0, len(freq_vec))), 0

    subset_df = all_cycles.get_metric_dataframe(subset=True)
    subset_indices = subset_df['index'].values
    all_cycles_inds = get_cycle_inds(all_cycles, subset_indices)
    cycle_bounds = _cycle_bounds_from_inds(all_cycles_inds)
    if cycle_bounds.size == 0:
        return np.zeros((0, len(freq_vec))), 0

    # 2) supra-theta signal (sum of IMFs > 12 Hz)
    sub_mask, theta_mask, supra_mask = tg_split(imf_centers_hz, theta_band)
    supra_theta_sig = imf[:, supra_mask].sum(axis=1) if np.any(supra_mask) else np.zeros(imf.shape[0])

    # 3) Morlet amplitude TFR (15–140 Hz) and optional normalization
    tf_amp = _morlet_amp_spectrogram(supra_theta_sig, fs, freq_vec, w=morlet_w)  # [n_freq, n_time]
    if normalize == 'zscore':
        tf_use = zscore(tf_amp, axis=1)  # per-freq z across time
    elif normalize == 'none':
        tf_use = tf_amp
    else:
        raise ValueError("normalize must be 'zscore' or 'none'.")

    # 4) Per-cycle FPP -> per-cycle spectral signature (mean across phase bins)
    sigs = []
    for b in cycle_bounds:
        # fpp_single: [1, n_freq, n_bins]  -> squeeze -> [n_freq, n_bins]
        fpp_single = bin_tf_to_fpp(b, tf_use, bin_count=n_bins)
        fpp_single = np.squeeze(fpp_single, axis=0)
        sig_single = np.nanmean(fpp_single, axis=1)  # collapse phase bins
        sigs.append(sig_single)

    if len(sigs) == 0:
        return np.zeros((0, len(freq_vec))), 0
    return np.vstack(sigs), len(sigs)

def extract_data_for_rat_fppsig(rat_id):
    base_path = '/Users/amir/Desktop/for Abdel/RGS/DatabyCondition'
    fs = 1000

    cfg = globals().get("RGSconfig", globals().get("config", None))
    if cfg is None:
        raise RuntimeError("Define RGSconfig (preferred) or config before calling this function.")

    all_combined_waveforms = pd.DataFrame()
    all_combined_trials = pd.DataFrame()
    all_phasic_fpps, all_tonic_fpps = [], []
    all_phasic_time_sigs, all_tonic_time_sigs = [], []

    # Support both HC and older CG naming
    preferred_conditions = ["HomeCageHC", "RandomCon", "OverlappingOR", "StableCondOD", "HomeCageCG"]
    conditions = [c for c in preferred_conditions if os.path.isdir(os.path.join(base_path, c))]
    if not conditions:
        print(f"No condition folders found under: {base_path}")
        return None, None

    folder_re = re.compile(r'^OS_Ephys_RGS14_Rat(\d+)_\d+_SD(\d+)_([\w-]+)_([\d-]+)$')

    for condition_folder in conditions:
        condition_path = os.path.join(base_path, condition_folder)

        # Only keep folders for this rat_id (avoids mismatch spam)
        rat_folders = []
        for f in os.listdir(condition_path):
            full = os.path.join(condition_path, f)
            if not os.path.isdir(full):
                continue
            m = folder_re.match(f)
            if not m:
                continue
            if m.group(1) == str(rat_id):
                rat_folders.append((f, m))

        if not rat_folders:
            continue

        for rat_folder, m in sorted(rat_folders, key=lambda x: x[0]):
            print(f"Processing rat folder: {rat_folder}")
            rat_path = os.path.join(condition_path, rat_folder)

            sd_number = m.group(2)
            condition = m.group(3)
            date_part = m.group(4)

            post_trial_folders = [
                f for f in os.listdir(rat_path)
                if os.path.isdir(os.path.join(rat_path, f)) and re.search(r'Post[-_]Trial\d+', f)
            ]
            post_trial_folders = sorted(post_trial_folders)

            for post_trial_folder in post_trial_folders:
                print(f"Processing post-trial folder: {post_trial_folder}")
                trial_path = os.path.join(rat_path, post_trial_folder)

                lfp_file, state_file = None, None
                for file_name in os.listdir(trial_path):
                    low = file_name.lower()
                    if ('hpc_100' in low) and low.endswith('.mat'):
                        lfp_file = os.path.join(trial_path, file_name)
                    elif ('states' in low) and low.endswith('.mat'):
                        state_file = os.path.join(trial_path, file_name)

                if not lfp_file or not state_file:
                    print(f"Missing LFP or state file in {trial_path}. Skipping...")
                    continue

                trial_number_match = re.search(r'Post[-_]Trial(\d+)', post_trial_folder)
                if not trial_number_match:
                    print(f"Unable to extract trial number from folder name: {post_trial_folder}. Skipping...")
                    continue
                trial_number = int(trial_number_match.group(1))

                try:
                    lfpHPC, hypno, _ = get_data(lfp_file, state_file)
                    try:
                        phasic_interval, tonic_interval, lfp_raw = extract_pt_intervals(lfpHPC, hypno, fs=1000)
                    except ValueError:
                        print(f"No REM sleep found in {post_trial_folder} (Rat {rat_id}, Condition {condition}).")
                        phasic_interval, tonic_interval, lfp_raw = [[], [], []]

                    if not (phasic_interval and tonic_interval):
                        continue

                    tonic_imfs, tonic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, tonic_interval, cfg, return_imfs_freqs=True)
                    phasic_imfs, phasic_freqs, _ = extract_imfs_by_pt_intervals(
                        lfp_raw, fs, phasic_interval, cfg, return_imfs_freqs=True)

                    phasic_waveforms, phasic_trials, phasic_fpps = extract_cycle_info(phasic_imfs, phasic_freqs)
                    tonic_waveforms, tonic_trials, tonic_fpps = extract_cycle_info(tonic_imfs, tonic_freqs)
                    all_phasic_fpps.extend(phasic_fpps)
                    all_tonic_fpps.extend(tonic_fpps)

                    for seg_imf, seg_freqs in zip(phasic_imfs, phasic_freqs):
                        sigs, _ = spectral_signatures_from_fpp_for_segment(
                            seg_imf, seg_freqs, fs,
                            theta_band=(5, 12),
                            freq_vec=np.arange(15, 141, 1),
                            morlet_w=6.0,
                            n_bins=19,
                            normalize='zscore'
                        )
                        all_phasic_time_sigs.append(sigs)

                    for seg_imf, seg_freqs in zip(tonic_imfs, tonic_freqs):
                        sigs, _ = spectral_signatures_from_fpp_for_segment(
                            seg_imf, seg_freqs, fs,
                            theta_band=(5, 12),
                            freq_vec=np.arange(15, 141, 1),
                            morlet_w=6.0,
                            n_bins=19,
                            normalize='zscore'
                        )
                        all_tonic_time_sigs.append(sigs)

                    for df in [phasic_waveforms, phasic_trials]:
                        df['rat_id'] = rat_id
                        df['condition'] = condition
                        df['trial'] = trial_number
                        df['cycle_type'] = 'phasic'
                        df['SD'] = sd_number
                        df['date'] = date_part

                    for df in [tonic_waveforms, tonic_trials]:
                        df['rat_id'] = rat_id
                        df['condition'] = condition
                        df['trial'] = trial_number
                        df['cycle_type'] = 'tonic'
                        df['SD'] = sd_number
                        df['date'] = date_part

                    all_combined_waveforms = pd.concat(
                        [all_combined_waveforms, phasic_waveforms, tonic_waveforms], ignore_index=True)
                    all_combined_trials = pd.concat(
                        [all_combined_trials, phasic_trials, tonic_trials], ignore_index=True)

                except FileNotFoundError:
                    print(f"Data not found in {trial_path}. Skipping...")

    if all_combined_waveforms.empty:
        print(f"No data extracted for Rat {rat_id}.")
        return None, None

    return (all_combined_waveforms, all_combined_trials,
            all_phasic_fpps, all_tonic_fpps,
            all_phasic_time_sigs, all_tonic_time_sigs)


In [ ]:
import pickle
import pandas as pd
from pathlib import Path

# ===== CONFIG =====
rat_ids  = [1, 2, 3, 4, 6, 7, 8, 9]
out_root = Path("./fppsig_rgs_10_140_fpp")
out_root.mkdir(parents=True, exist_ok=True)

# toggle between saving separately or as one pickle
save_all_together = False   # ← True = all_objects.pkl, False = waveforms_df + trials_df + lists.pkl

def _save_df(df: pd.DataFrame, path_no_ext: Path) -> str:
    """Save DataFrame as Parquet if possible; else CSV. Return saved path."""
    try:
        p = path_no_ext.with_suffix(".parquet")
        df.to_parquet(str(p), index=False)
        return str(p)
    except Exception:
        p = path_no_ext.with_suffix(".csv")
        df.to_csv(str(p), index=False)
        return str(p)

manifest_rows = []

for rid in rat_ids:
    print(f"\n=== Rat {rid} ===")
    res = extract_data_for_rat_fppsig(str(rid))
    if not isinstance(res, tuple) or len(res) != 6:
        print(f"  Skipping Rat {rid}: unexpected return value.")
        continue

    (waveforms_df,
     trials_df,
     all_phasic_FPPs,
     all_tonic_FPPs,
     phasic_time_signatures,
     tonic_time_signatures) = res

    if waveforms_df is None or trials_df is None:
        print(f"  Skipping Rat {rid}: no data.")
        continue

    rat_dir = out_root / f"Rat{rid}"
    rat_dir.mkdir(parents=True, exist_ok=True)

    if save_all_together:
        # ----- one combined file -----
        all_path = rat_dir / "all_objects.pkl"
        with open(all_path, "wb") as f:
            pickle.dump(
                dict(
                    waveforms_df=waveforms_df,
                    trials_df=trials_df,
                    all_phasic_FPPs=all_phasic_FPPs,
                    all_tonic_FPPs=all_tonic_FPPs,
                    phasic_time_signatures=phasic_time_signatures,
                    tonic_time_signatures=tonic_time_signatures
                ),
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )
        wf_path = tr_path = lists_path = None
        print(f"  Saved everything → {all_path}")

    else:
        # ----- separate files -----
        wf_path = _save_df(waveforms_df, rat_dir / "waveforms_df")
        tr_path = _save_df(trials_df,   rat_dir / "trials_df")

        lists_path = rat_dir / "lists.pkl"
        with open(lists_path, "wb") as f:
            pickle.dump(
                dict(
                    all_phasic_FPPs=all_phasic_FPPs,
                    all_tonic_FPPs=all_tonic_FPPs,
                    phasic_time_signatures=phasic_time_signatures,
                    tonic_time_signatures=tonic_time_signatures
                ),
                f,
                protocol=pickle.HIGHEST_PROTOCOL
            )
        print(f"  Saved separate files → {rat_dir}")

    # --- manifest info ---
    n_phasic_segments = len(phasic_time_signatures)
    n_tonic_segments  = len(tonic_time_signatures)
    n_phasic_cycles   = int(sum((arr.shape[0] for arr in phasic_time_signatures if hasattr(arr, "shape")), 0))
    n_tonic_cycles    = int(sum((arr.shape[0] for arr in tonic_time_signatures  if hasattr(arr, "shape")), 0))

    manifest_rows.append({
        "rat_id": rid,
        "n_phasic_segments": n_phasic_segments,
        "n_tonic_segments": n_tonic_segments,
        "n_phasic_cycles_total": n_phasic_cycles,
        "n_tonic_cycles_total": n_tonic_cycles,
        "waveforms_df_path": wf_path,
        "trials_df_path": tr_path,
        "lists_pickle_path": str(lists_path) if not save_all_together else None,
        "all_objects_pickle_path": str(all_path) if save_all_together else None
    })

# save manifest
manifest_df = pd.DataFrame(manifest_rows).sort_values("rat_id").reset_index(drop=True)
manifest_path = out_root / "manifest.csv"
manifest_df.to_csv(manifest_path, index=False)
print(f"\nManifest saved to: {manifest_path}")
print(manifest_df)

## laod all rats

In [ ]:
import pickle
import pandas as pd
from pathlib import Path

def load_rat_data(rid, out_root=Path("./fppsig_rgs_10_140_fpp")):
    """Load all data objects for a single rat."""
    rat_dir = out_root / f"Rat{rid}"

    # Option 1: combined pickle
    all_pkl = rat_dir / "all_objects.pkl"
    if all_pkl.exists():
        with open(all_pkl, "rb") as f:
            return pickle.load(f)

    # Option 2: separate files
    def _load_df(name):
        for ext in [".parquet", ".csv"]:
            p = rat_dir / f"{name}{ext}"
            if p.exists():
                return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
        return None

    waveforms_df = _load_df("waveforms_df")
    trials_df = _load_df("trials_df")

    lists_pkl = rat_dir / "lists.pkl"
    if lists_pkl.exists():
        with open(lists_pkl, "rb") as f:
            lists = pickle.load(f)
    else:
        lists = {}

    return dict(waveforms_df=waveforms_df, trials_df=trials_df, **lists)


def load_all_rats(out_root=Path("./fppsig_rgs_10_140_fpp")):
    """Load all rats listed in manifest.csv into a dict."""
    manifest_path = out_root / "manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found at {manifest_path}")

    manifest = pd.read_csv(manifest_path)
    rat_ids = manifest["rat_id"].tolist()

    all_data = {}
    for rid in rat_ids:
        print(f"Loading Rat {rid} ...", end=" ")
        try:
            all_data[rid] = load_rat_data(rid, out_root)
            print("✅ done")
        except Exception as e:
            print(f"⚠️ failed ({e})")

    return all_data

In [ ]:
all_rats = load_all_rats()

# extract tSCs - all rats

## run ICA/PCA

In [ ]:
import numpy as np
import pandas as pd
import pickle
from pathlib import Path
from sklearn.decomposition import PCA, FastICA

try:
    from scipy.ndimage import gaussian_filter1d
    _HAS_SCI = True
except Exception:
    _HAS_SCI = False


def load_rat_data(rid, out_root=Path("./fppsig_rgs_10_140_fpp")):
    """Load all data objects for a single rat."""
    rat_dir = out_root / f"Rat{rid}"
    
    # Option 1: combined pickle
    all_pkl = rat_dir / "all_objects.pkl"
    if all_pkl.exists():
        with open(all_pkl, "rb") as f:
            return pickle.load(f)
    
    # Option 2: separate files
    def _load_df(name):
        for ext in [".parquet", ".csv"]:
            p = rat_dir / f"{name}{ext}"
            if p.exists():
                return pd.read_parquet(p) if ext == ".parquet" else pd.read_csv(p)
        return None
    
    waveforms_df = _load_df("waveforms_df")
    trials_df = _load_df("trials_df")
    
    lists_pkl = rat_dir / "lists.pkl"
    if lists_pkl.exists():
        with open(lists_pkl, "rb") as f:
            lists = pickle.load(f)
    else:
        lists = {}
    
    return dict(waveforms_df=waveforms_df, trials_df=trials_df, **lists)


def load_all_rats(out_root=Path("./fppsig_rgs_10_140_fpp")):
    """Load all rats listed in manifest.csv into a dict."""
    manifest_path = out_root / "manifest.csv"
    if not manifest_path.exists():
        raise FileNotFoundError(f"Manifest not found at {manifest_path}")
    
    manifest = pd.read_csv(manifest_path)
    rat_ids = manifest["rat_id"].tolist()
    
    all_data = {}
    for rid in rat_ids:
        print(f"Loading Rat {rid} ...", end=" ")
        try:
            all_data[rid] = load_rat_data(rid, out_root)
            print("✅ done")
        except Exception as e:
            print(f"⚠️ failed ({e})")
    
    return all_data


def run_multi_rat_tSC_pipeline(all_rats_data, 
                                n_pca=5, 
                                freq_vec=np.arange(15, 141, 1),
                                zscore_features=True, 
                                mad_k=2.0,
                                weighted_alpha=0.4,
                                ignore_edge_bins=1):
    """
    Run tSC pipeline on combined data from all rats.
    
    Parameters:
    -----------
    all_rats_data : dict
        Dictionary with rat_id as keys, each containing:
        - phasic_time_signatures: list of arrays
        - tonic_time_signatures: list of arrays
    
    Returns:
    --------
    cycles_df : pd.DataFrame
        DataFrame with all cycles from all rats, including rat_id and all tSC metrics
    tSC_results : dict
        Dictionary containing PCA/ICA components and other analysis results
    """
    
    # ============ 1) Combine data from all rats ============
    print("Combining data from all rats...")
    
    all_phasic_sigs = []
    all_tonic_sigs = []
    rat_metadata = []  # track which rat each signature belongs to
    
    for rat_id, data in all_rats_data.items():
        phasic_sigs = data.get('phasic_time_signatures', [])
        tonic_sigs = data.get('tonic_time_signatures', [])
        
        # Track rat_id for each segment
        for seg in phasic_sigs:
            if isinstance(seg, np.ndarray) and seg.size > 0:
                all_phasic_sigs.append(seg)
                rat_metadata.append({'rat_id': rat_id, 'cycle_type': 'phasic'})
        
        for seg in tonic_sigs:
            if isinstance(seg, np.ndarray) and seg.size > 0:
                all_tonic_sigs.append(seg)
                rat_metadata.append({'rat_id': rat_id, 'cycle_type': 'tonic'})
    
    print(f"  Total phasic segments: {len(all_phasic_sigs)}")
    print(f"  Total tonic segments: {len(all_tonic_sigs)}")
    
    # ============ 2) Flatten all signatures ============
    def _flatten_sig_list(sig_list, metadata_list, label):
        rows, meta = [], []
        seg_idx = 0
        
        for sig_group in sig_list:
            # Find corresponding metadata
            rat_meta = [m for m in metadata_list if m['cycle_type'] == label][seg_idx] if seg_idx < len([m for m in metadata_list if m['cycle_type'] == label]) else {}
            
            if not isinstance(sig_group, np.ndarray) or sig_group.size == 0:
                continue
            
            mask = np.isfinite(sig_group).all(axis=1)
            Xi = sig_group[mask]
            
            if Xi.size == 0:
                continue
            
            rows.append(Xi)
            for j in range(Xi.shape[0]):
                meta.append({
                    "interval_idx": seg_idx,
                    "cycle_idx_in_interval": j,
                    "cycle_type": label,
                    "rat_id": rat_meta.get('rat_id', None)
                })
            
            seg_idx += 1
        
        if len(rows) == 0:
            return np.zeros((0, len(freq_vec))), []
        
        return np.vstack(rows), meta
    
    # Separate metadata by cycle type
    phasic_meta = [m for m in rat_metadata if m['cycle_type'] == 'phasic']
    tonic_meta = [m for m in rat_metadata if m['cycle_type'] == 'tonic']
    
    X_phasic, meta_phasic = _flatten_sig_list(all_phasic_sigs, phasic_meta, "phasic")
    X_tonic, meta_tonic = _flatten_sig_list(all_tonic_sigs, tonic_meta, "tonic")
    
    X = np.vstack([X_phasic, X_tonic])
    meta = meta_phasic + meta_tonic
    
    if X.shape[0] == 0:
        raise RuntimeError("No valid cycles to analyze.")
    
    print(f"Total cycles combined: {X.shape[0]}")
    print(f"  Phasic: {X_phasic.shape[0]}")
    print(f"  Tonic: {X_tonic.shape[0]}")
    
    # ============ 3) Feature z-score ============
    if zscore_features:
        mu = X.mean(axis=0, keepdims=True)
        sd = X.std(axis=0, ddof=1, keepdims=True) + 1e-12
        Xz = (X - mu) / sd
    else:
        Xz = X
    
    # ============ 4) Smoothing helper ============
    def _smooth_rows(mat, sigma_hz, mode="reflect"):
        if _HAS_SCI:
            return gaussian_filter1d(mat, sigma=float(sigma_hz), axis=1, mode=mode)
        
        sigma = float(sigma_hz)
        rad = int(np.ceil(4 * sigma))
        kx = np.arange(-rad, rad + 1)
        ker = np.exp(-(kx**2) / (2 * sigma**2))
        ker /= ker.sum()
        pad = rad
        out = np.empty_like(mat)
        
        for i in range(mat.shape[0]):
            row = mat[i]
            if mode == "reflect":
                row_pad = np.r_[row[pad:0:-1], row, row[-2:-pad-2:-1]]
            elif mode == "constant":
                row_pad = np.r_[np.zeros(pad), row, np.zeros(pad)]
            else:
                row_pad = np.r_[row[pad:0:-1], row, row[-2:-pad-2:-1]]
            out[i] = np.convolve(row_pad, ker, mode="same")[pad:-pad]
        
        return out
    
    # ============ 5) Create smoothed versions ============
    Xz_smooth5_ref = _smooth_rows(Xz, 5.0, mode="reflect")
    Xz_smooth10_ref = _smooth_rows(Xz, 10.0, mode="reflect")
    Xz_smooth5_pad = _smooth_rows(Xz, 5.0, mode="constant")
    Xz_smooth10_pad = _smooth_rows(Xz, 10.0, mode="constant")
    
    # ============ 6) PCA → ICA ============
    print("Running PCA...")
    pca = PCA(n_components=n_pca, random_state=42)
    Z = pca.fit_transform(Xz)
    
    print("Running ICA...")
    ica = FastICA(n_components=n_pca, random_state=42, max_iter=1000, tol=1e-3)
    S_latent = ica.fit_transform(Z)
    W_freq = ica.components_ @ pca.components_
    
    # ============ 7) Sign fix ============
    mean_src = S_latent.mean(axis=0, keepdims=True)
    signs = np.sign(mean_src)
    signs[signs == 0] = 1
    S_latent *= signs
    W_freq *= signs.T
    
    # ============ 8) Compute strengths ============
    strengths = Xz @ W_freq.T
    abs_strengths = np.abs(strengths)
    
    # ============ 9) Thresholds ============
    def _mad_threshold(x, k=2.0):
        med = np.median(x)
        mad = np.median(np.abs(x - med)) + 1e-12
        return med + k * (mad / 0.6745)
    
    thr_per_comp = np.array([_mad_threshold(abs_strengths[:, k], k=mad_k) 
                              for k in range(n_pca)])
    
    labels_0based = np.argmax(abs_strengths, axis=1)
    
    # ============ 10) Winner-strong mask ============
    strong_mask = np.zeros_like(labels_0based, dtype=bool)
    for k in range(n_pca):
        strong_mask |= (labels_0based == k) & (abs_strengths[:, k] >= thr_per_comp[k])
    
    # ============ 11) Paper-style per-component strong ============
    strong_mask_matrix = abs_strengths >= thr_per_comp[None, :]
    tsc_n_strong = strong_mask_matrix.sum(axis=1)
    tsc_any_strong = tsc_n_strong > 0
    tsc_strong_components = [list(np.nonzero(row)[0] + 1) for row in strong_mask_matrix]
    tsc_strong_components_str = [",".join(map(str, lst)) if len(lst) else "" 
                                  for lst in tsc_strong_components]
    tsc_exclusive_label = np.where(tsc_n_strong == 1,
                                    (np.argmax(strong_mask_matrix, axis=1) + 1),
                                    np.nan)
    
    # ============ 12) Mode extraction helper ============
    def _mode_from_mat(mat):
        L, R = ignore_edge_bins, (mat.shape[1] - ignore_edge_bins)
        if R <= L:
            idx = np.argmax(mat, axis=1)
        else:
            idx = np.argmax(mat[:, L:R], axis=1) + L
        return freq_vec[idx]
    
    # ============ 13) Extract modes ============
    mode_freq_hz_featZ = _mode_from_mat(Xz)
    mode_freq_hz_featZ_smooth = _mode_from_mat(Xz_smooth5_ref)
    mode_freq_hz_featZ_smooth10 = _mode_from_mat(Xz_smooth10_ref)
    mode_freq_hz_featZ_smooth5_pad = _mode_from_mat(Xz_smooth5_pad)
    mode_freq_hz_featZ_smooth10_pad = _mode_from_mat(Xz_smooth10_pad)
    
    # ============ 14) Strong modes ============
    def _mode_strong(mat):
        modes = np.full(mat.shape[0], np.nan)
        modes[strong_mask] = _mode_from_mat(mat[strong_mask])
        return modes
    
    mode_freq_hz_featZ_strong = _mode_strong(Xz)
    mode_freq_hz_featZ_smooth_strong = _mode_strong(Xz_smooth5_ref)
    mode_freq_hz_featZ_smooth10_strong = _mode_strong(Xz_smooth10_ref)
    
    # ============ 15) Weighted modes ============
    def mode_from_feature_z_weighted(Xz_like, W_freq, labels_0based, freq_vec,
                                      avoid_edge_bins=1, alpha=None):
        n_cycles, n_freq = Xz_like.shape
        lo = avoid_edge_bins
        hi = n_freq - avoid_edge_bins
        use_slice = slice(lo, hi) if hi > lo else slice(0, n_freq)
        modes = np.empty(n_cycles, dtype=float)
        
        for i in range(n_cycles):
            k = int(labels_0based[i])
            w = np.abs(W_freq[k]).copy()
            
            if alpha is not None:
                thr = alpha * np.max(w)
                m = (w >= thr)
                if m.sum() >= 3:
                    w[~m] = 0.0
            
            y = Xz_like[i] * w
            yseg = y[use_slice]
            idx_rel = int(np.argmax(yseg))
            idx = (lo + idx_rel) if hi > lo else idx_rel
            modes[i] = freq_vec[idx]
        
        return modes
    
    mode_freq_hz_featZ_weighted = mode_from_feature_z_weighted(
        Xz, W_freq, labels_0based, freq_vec,
        avoid_edge_bins=ignore_edge_bins, alpha=weighted_alpha
    )
    
    # ============ 16) Within-cycle-Z + weights ============
    eps = 1e-12
    
    def tsc_weighted_mode_freq(X, W_freq, labels_0based, freq_vec,
                                avoid_edge_bins=1, alpha=0.4):
        n_cycles, n_freq = X.shape
        modes = np.empty(n_cycles, dtype=float)
        lo = avoid_edge_bins
        hi = n_freq - avoid_edge_bins
        use_slice = slice(lo, hi) if hi > lo else slice(0, n_freq)
        
        for i in range(n_cycles):
            k = int(labels_0based[i])
            x = X[i]
            xz = (x - x.mean()) / (x.std(ddof=1) + eps)
            w = np.abs(W_freq[k]).copy()
            
            if alpha is not None:
                thr = alpha * np.max(w)
                mask = w >= thr
                if mask.sum() >= 3:
                    w[~mask] = 0.0
            
            y = xz * w
            yseg = y[use_slice]
            idx_rel = int(np.argmax(yseg))
            idx = (lo + idx_rel) if hi > lo else idx_rel
            modes[i] = freq_vec[idx]
        
        return modes
    
    mode_freq_hz_proj = tsc_weighted_mode_freq(
        X, W_freq, labels_0based, freq_vec,
        avoid_edge_bins=ignore_edge_bins, alpha=weighted_alpha
    )
    
    # ============ 17) Package results ============
    cycles_df = pd.DataFrame(meta)
    cycles_df["tSC_label"] = labels_0based + 1
    cycles_df["mode_freq_hz_featZ"] = mode_freq_hz_featZ
    cycles_df["mode_freq_hz_featZ_smooth"] = mode_freq_hz_featZ_smooth
    cycles_df["mode_freq_hz_featZ_smooth10"] = mode_freq_hz_featZ_smooth10
    cycles_df["mode_freq_hz_featZ_smooth5_pad"] = mode_freq_hz_featZ_smooth5_pad
    cycles_df["mode_freq_hz_featZ_smooth10_pad"] = mode_freq_hz_featZ_smooth10_pad
    cycles_df["mode_freq_hz_featZ_strong"] = mode_freq_hz_featZ_strong
    cycles_df["mode_freq_hz_featZ_smooth_strong"] = mode_freq_hz_featZ_smooth_strong
    cycles_df["mode_freq_hz_featZ_smooth10_strong"] = mode_freq_hz_featZ_smooth10_strong
    cycles_df["mode_freq_hz_featZw"] = mode_freq_hz_featZ_weighted
    cycles_df["mode_freq_hz_proj"] = mode_freq_hz_proj
    
    # Per-component strengths/flags
    for k in range(n_pca):
        cycles_df[f"tSC{k+1}_strength"] = strengths[:, k]
        cycles_df[f"tSC{k+1}_strong"] = (abs_strengths[:, k] >= thr_per_comp[k])
    
    # Paper-style summaries
    cycles_df["tSC_any_strong"] = tsc_any_strong
    cycles_df["tSC_n_strong"] = tsc_n_strong
    cycles_df["tSC_strong_components"] = tsc_strong_components
    cycles_df["tSC_strong_components_str"] = tsc_strong_components_str
    cycles_df["tSC_winner_strong"] = strong_mask
    cycles_df["tSC_exclusive_label"] = tsc_exclusive_label
    
    tSC_results = {
        "freq_vec": freq_vec,
        "pca": pca,
        "ica": ica,
        "weights_freq": W_freq,
        "strengths": strengths,
        "latent_sources_S": S_latent,
        "thresholds_abs_strength": thr_per_comp,
        "X_cycles": X,
        "X_cycles_featZ": Xz,
        "X_cycles_featZ_smooth": Xz_smooth5_ref,
        "X_cycles_featZ_smooth10": Xz_smooth10_ref,
        "X_cycles_featZ_smooth5_pad": Xz_smooth5_pad,
        "X_cycles_featZ_smooth10_pad": Xz_smooth10_pad,
        "strong_mask_matrix": strong_mask_matrix,
        "winner_strong_mask": strong_mask,
        "meta": meta
    }
    
    # ============ 18) Print summary ============
    print("\n=== Multi-Rat PCA/ICA Complete ===")
    print(f"Total cycles: {X.shape[0]}")
    print(f"Mode — featZ (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ']):.2f} Hz")
    print(f"Mode — featZ strong (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ_strong']):.2f} Hz")
    print(f"Mode — featZ 5Hz reflect strong (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ_smooth_strong']):.2f} Hz")
    print(f"Mode — featZ 10Hz reflect strong (median): {np.nanmedian(cycles_df['mode_freq_hz_featZ_smooth10_strong']):.2f} Hz")
    
    # Per-rat summary
    print("\nPer-rat cycle counts:")
    rat_counts = cycles_df.groupby('rat_id').size()
    for rat_id, count in rat_counts.items():
        print(f"  Rat {rat_id}: {count} cycles")
    
    return cycles_df, tSC_results


In [ ]:
# Run multi-rat tSC pipeline
print("\nRunning multi-rat tSC pipeline...")
cycles_df, tSC_results = run_multi_rat_tSC_pipeline(
        all_rats,
        n_pca=5,
        freq_vec=np.arange(15, 141, 1),
        zscore_features=True,
        mad_k=2.0,
        weighted_alpha=0.4,
        ignore_edge_bins=1
    )
    
# Save results
# output_dir = Path("./multi_rat_tSC_results")
# output_dir.mkdir(parents=True, exist_ok=True)
    
# # Save cycles dataframe
# cycles_df.to_csv(output_dir / "cycles_df_all_rats.csv", index=False)
# print(f"\nSaved cycles_df to {output_dir / 'cycles_df_all_rats.csv'}")
    
# # Save tSC results as pickle
# with open(output_dir / "tSC_results_all_rats.pkl", "wb") as f:
#     pickle.dump(tSC_results, f, protocol=pickle.HIGHEST_PROTOCOL)
# print(f"Saved tSC_results to {output_dir / 'tSC_results_all_rats.pkl'}")
    
print("\n✅ Analysis complete!")

## Analysis

In [ ]:
# ============================================================
# E) SMALL UTILITIES
# ============================================================

def plot_tSC_weights(weights, freqs):
    plt.figure(figsize=(12, 6))
    
    # Color palette avoiding green - using blues, reds, purples, oranges, browns
    color_palette = [
        '#1f77b4',  # blue
        '#ff7f0e',  # orange  
        '#9467bd',  # purple
        '#8c564b',  # brown
        '#e377c2',  # pink
        '#7f7f7f',  # gray
        '#bcbd22',  # olive (not green)
        '#17becf',  # cyan
        '#d62728',  # red
        '#ff9896'   # light red
    ]
    
    # Extend palette if needed
    while len(color_palette) < weights.shape[0]:
        color_palette.extend(color_palette)
    
    for k in range(weights.shape[0]):
        w = weights[k]
        pk = freqs[np.argmax(np.abs(w))]
        color = color_palette[k % len(color_palette)]
        plt.plot(freqs, w, label=f"tSC {k+1} (peak {pk:.1f} Hz)", color=color)
        plt.scatter([pk], [w[np.argmax(np.abs(w))]], s=40, color=color)
    
    plt.xlabel("Frequency (Hz)")
    plt.ylabel("Weight")
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

def plot_mode_freq_histograms_per_component(df, col="mode_freq_hz_proj",
                                            freq_bins=np.geomspace(15, 310, 24), ymax=0.25):
    comps = sorted(df["tSC_label"].unique())
    for comp in comps:
        sub = df[df["tSC_label"] == comp]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
        for j, ctype in enumerate(["phasic", "tonic"]):
            ax = axes[j]
            vals = sub.loc[sub["cycle_type"] == ctype, col].to_numpy()
            vals = vals[np.isfinite(vals)]
            if vals.size > 0:
                weights = np.ones_like(vals, dtype=float) / vals.size
                ax.hist(vals, bins=freq_bins, weights=weights, alpha=0.8)
            ax.set_xlim(freq_bins[0], freq_bins[-1]); ax.set_ylim(0, ymax)
            ax.set_xlabel("Mode frequency (Hz)"); ax.set_ylabel("Probability")
            ax.set_title(f"tSC {comp} — {ctype.capitalize()}")
        plt.tight_layout(); plt.show()

def print_cycle_counts(cycles_df):
    counts = cycles_df["cycle_type"].value_counts()
    print(f"Phasic cycles: {counts.get('phasic', 0)}")
    print(f"Tonic  cycles: {counts.get('tonic', 0)}")

def plot_cycle_signature(cycle_idx, X_raw, X_z, freq_vec):
    if cycle_idx < 0 or cycle_idx >= X_raw.shape[0]:
        print(f"Invalid cycle_idx {cycle_idx}, max = {X_raw.shape[0]-1}"); return
    plt.figure(figsize=(12,4))
    plt.subplot(1,2,1); plt.plot(freq_vec, X_raw[cycle_idx], color="black")
    plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (a.u.)"); plt.title(f"Cycle {cycle_idx} — Raw signature")
    plt.subplot(1,2,2); plt.plot(freq_vec, X_z[cycle_idx], color="red")
    plt.xlabel("Frequency (Hz)"); plt.ylabel("Z-scored Power"); plt.title(f"Cycle {cycle_idx} — Z-scored signature")
    plt.tight_layout(); plt.show()

def plot_multiple_cycle_signatures(cycles_df, X_raw, X_z, freq_vec, n_cycles=10, cycle_type="phasic"):
    idxs = cycles_df.index[cycles_df["cycle_type"] == cycle_type].to_numpy()
    if len(idxs) == 0:
        print(f"No cycles of type '{cycle_type}' found."); return
    idxs = idxs[:n_cycles]
    for i, idx in enumerate(idxs, 1):
        plt.figure(figsize=(12,4))
        plt.subplot(1,2,1); plt.plot(freq_vec, X_raw[idx], color="black")
        plt.xlabel("Frequency (Hz)"); plt.ylabel("Power (a.u.)"); plt.title(f"{cycle_type.capitalize()} cycle {i} (raw)")
        plt.subplot(1,2,2); plt.plot(freq_vec, X_z[idx], color="red")
        plt.xlabel("Frequency (Hz)"); plt.ylabel("Z-scored Power"); plt.title(f"{cycle_type.capitalize()} cycle {i} (z-scored)")
        plt.tight_layout(); plt.show()

In [ ]:
print_cycle_counts(cycles_df)
plot_tSC_weights(tSC_results["weights_freq"], tSC_results["freq_vec"])

In [ ]:
def plot_mode_freq_histograms_per_component(
        df,
        col="mode_freq_hz_featZ_smooth",   # can be str or list[str]
        freq_bins=np.linspace(15, 140, 20),
        ymax=0.25,
        peaks_hz=None,              # optional: array of component peak freqs
        weights=None, freq_vec=None # or pass weights+freq_vec to compute peaks
    ):
    """
    Plot per-component histograms for phasic/tonic.
    `col` can be a single column name (str) or a list of column names to overlay.

    Recognized columns include:
      'mode_freq_hz_featZ', 'mode_freq_hz_featZ_smooth', 'mode_freq_hz_featZ_smooth10',
      'mode_freq_hz_featZ_smooth5_pad', 'mode_freq_hz_featZ_smooth10_pad',
      'mode_freq_hz_featZw', 'mode_freq_hz_proj'
    """
    # allow col to be list for overlays
    cols = [col] if isinstance(col, str) else list(col)
    palette = plt.cm.tab10(np.linspace(0, 1, max(10, len(cols))))

    comps = sorted(df["tSC_label"].dropna().unique().astype(int))

    # Derive peaks if not provided but weights+freq_vec are available
    if peaks_hz is None and (weights is not None) and (freq_vec is not None):
        weights = np.asarray(weights)
        freq_vec = np.asarray(freq_vec)
        peak_idx = np.argmax(np.abs(weights), axis=1)
        computed_peaks = freq_vec[peak_idx]
        peaks_map = {k+1: computed_peaks[k] for k in range(weights.shape[0])}
    else:
        peaks_map = {}
        if peaks_hz is not None:
            peaks_hz = np.asarray(peaks_hz)
            for i, p in enumerate(peaks_hz, start=1):
                peaks_map[i] = p

    for comp in comps:
        sub = df[df["tSC_label"] == comp]
        fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)

        for j, ctype in enumerate(["phasic", "tonic"]):
            ax = axes[j]
            for ci, c in enumerate(cols):
                vals = sub.loc[sub["cycle_type"] == ctype, c].to_numpy(dtype=float)
                vals = vals[np.isfinite(vals)]
                if vals.size > 0:
                    weights_norm = np.ones_like(vals, dtype=float) / vals.size
                    ax.hist(vals, bins=freq_bins, weights=weights_norm,
                            alpha=0.5, color=palette[ci], label=c)
            ax.set_xlim(freq_bins.min(), freq_bins.max())
            ax.set_ylim(0, ymax)
            ax.set_xlabel("Mode frequency (Hz)")
            if j == 0:
                ax.set_ylabel("Probability")
            ax.set_title(f"tSC {comp} — {ctype.capitalize()}")

            # vertical line at component peak
            if comp in peaks_map and np.isfinite(peaks_map[comp]):
                pk = float(peaks_map[comp])
                ax.axvline(pk, ls="--", lw=2, color="k", alpha=0.9)
                ax.text(pk, ymax*0.95, f"{pk:.1f} Hz", ha="center", va="top")

            if j == 1:
                ax.legend(fontsize=8)

        plt.tight_layout()
        plt.show()

In [ ]:
plot_mode_freq_histograms_per_component(
    cycles_df,
    col=["mode_freq_hz_featZ_smooth5_pad"],
    freq_bins=np.linspace(15, 141, 30),
    ymax=0.25,
    weights=tSC_results["weights_freq"],
    freq_vec=tSC_results["freq_vec"]
)

In [ ]:
# ============================================================
# Add PCA scores + "likelihoods" (probabilities) for PCA & ICA
# ============================================================
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ------- pull from your results -------
pca        = tSC_results["pca"]
ica        = tSC_results["ica"]
X_featZ    = tSC_results["X_cycles_featZ"]   # the same features the PCA was fit on
S          = tSC_results["strengths"]        # (n_cycles × n_pca) = ICA source strengths
evr        = pca.explained_variance_ratio_   # (n_pca,)
n_pca      = S.shape[1]

# safety
assert X_featZ.shape[0] == cycles_df.shape[0], "Row mismatch: X_featZ vs cycles_df"
assert S.shape[0]       == cycles_df.shape[0], "Row mismatch: S vs cycles_df"

# ------- recompute PCA scores (Z) for all cycles -------
Z = pca.transform(X_featZ)   # (n_cycles × n_pca)

# ------------------------------------------------------------
# helpers for probabilities ("likelihoods")
# ------------------------------------------------------------
def _softmax(a, axis=1, tau=1.0):
    # numerically-stable softmax with temperature
    a = np.asarray(a, dtype=float)
    if tau <= 0:
        tau = 1.0
    m = np.max(a, axis=axis, keepdims=True)
    e = np.exp((a - m) / tau)
    return e / (e.sum(axis=axis, keepdims=True) + 1e-12)

def _mad_scale(x, axis=0):
    # robust sigma ~ MAD/0.6745
    med = np.median(x, axis=axis, keepdims=True)
    mad = np.median(np.abs(x - med), axis=axis, keepdims=True)
    return (mad / 0.6745) + 1e-12

# ------------------------------------------------------------
# A) ICA/tSC probabilities
# ------------------------------------------------------------
# Option 1: plain |S|
S_abs = np.abs(S)

# Option 2 (recommended): scale each component by its robust std (MAD)
S_sigma = _mad_scale(S_abs, axis=0)              # (1 × n_pca)
S_abs_mad = S_abs / S_sigma                      # component-wise normalization

tau_ica = 1.0   # temperature for softmax (lower -> more "peaky")
probs_tSC_plain = _softmax(S_abs,     axis=1, tau=tau_ica)
probs_tSC_mad   = _softmax(S_abs_mad, axis=1, tau=tau_ica)

# choose which one to keep as your "official" probability
probs_tSC = probs_tSC_mad    # <-- pick MAD-normalized by default

# argmax label, confidence & entropy
tSC_top_idx   = np.argmax(probs_tSC, axis=1)              # 0-based
tSC_conf      = probs_tSC[np.arange(probs_tSC.shape[0]), tSC_top_idx]
tSC_entropy   = -(probs_tSC * np.log(probs_tSC + 1e-12)).sum(axis=1)

# ------------------------------------------------------------
# B) PCA probabilities
# ------------------------------------------------------------
# Two sensible flavors; pick one:
#  B1) variance-weighted |Z|   (|Z_k| * sqrt(EVR_k))
#  B2) variance-weighted power (Z_k^2 * EVR_k)
w_sqrt = np.sqrt(evr)[None, :]         # (1 × n_pca)
w_evr  = evr[None, :]                  # (1 × n_pca)

pca_score_mag = np.abs(Z) * w_sqrt     # B1
pca_score_pow = (Z**2)   * w_evr       # B2

tau_pca = 1.0
probs_PC_mag = _softmax(pca_score_mag, axis=1, tau=tau_pca)
probs_PC_pow = _softmax(pca_score_pow, axis=1, tau=tau_pca)

# choose your "official" PCA prob
probs_PC = probs_PC_mag

PC_top_idx = np.argmax(probs_PC, axis=1)
PC_conf    = probs_PC[np.arange(probs_PC.shape[0]), PC_top_idx]
PC_entropy = -(probs_PC * np.log(probs_PC + 1e-12)).sum(axis=1)

# ------------------------------------------------------------
# C) Write everything back into cycles_df
# ------------------------------------------------------------
# PCA scores
for k in range(n_pca):
    cycles_df[f"PC{k+1}_score"] = Z[:, k]

# PCA probabilities
for k in range(n_pca):
    cycles_df[f"PC{k+1}_prob"]      = probs_PC[:, k]
    cycles_df[f"PC{k+1}_prob_pow"]  = probs_PC_pow[:, k]   # optional alt flavor

cycles_df["PC_top"]       = (PC_top_idx + 1).astype(int)
cycles_df["PC_conf"]      = PC_conf
cycles_df["PC_entropy"]   = PC_entropy

# ICA/tSC probabilities
for k in range(n_pca):
    cycles_df[f"tSC{k+1}_prob"]        = probs_tSC[:, k]        # MAD-normalized (chosen)
    cycles_df[f"tSC{k+1}_prob_plain"]  = probs_tSC_plain[:, k]  # plain |S| softmax (reference)
    cycles_df[f"tSC{k+1}_zmad"]        = S_abs_mad[:, k]        # |S| normalized by MAD (diagnostic)

cycles_df["tSC_top"]      = (tSC_top_idx + 1).astype(int)
cycles_df["tSC_conf"]     = tSC_conf
cycles_df["tSC_entropy"]  = tSC_entropy

print("Added PCA scores + PCA/ICA probabilities to cycles_df")
print(cycles_df.filter(regex=r'^(PC\d+_score|PC\d+_prob|tSC\d+_prob|tSC_top|tSC_conf|PC_top|PC_conf)$').head())

# ------------------------------------------------------------
# D) (Optional) quick sanity plots
# ------------------------------------------------------------
def plot_prob_summaries(cycles_df, comp_type="tSC", n_comp=n_pca):
    fig, ax = plt.subplots(figsize=(6,4))
    conf = cycles_df[f"{comp_type}_conf"].to_numpy()
    ax.hist(conf[np.isfinite(conf)], bins=30)
    ax.set_xlabel(f"{comp_type} max probability (confidence)")
    ax.set_ylabel("Count")
    ax.set_title(f"{comp_type} confidence histogram")
    plt.tight_layout(); plt.show()

    # mean probability across cycles per component (should be ~uniform if labels balanced)
    means = [cycles_df[f"{comp_type}{k+1}_prob"].mean() for k in range(n_comp)]
    fig, ax = plt.subplots(figsize=(6,4))
    ax.bar(np.arange(1, n_comp+1), means)
    ax.set_xlabel(f"{comp_type} component")
    ax.set_ylabel("Mean probability")
    ax.set_title(f"Mean {comp_type} probability per component")
    plt.tight_layout(); plt.show()

plot_prob_summaries(cycles_df, comp_type="tSC", n_comp=n_pca)
plot_prob_summaries(cycles_df, comp_type="PC",  n_comp=n_pca)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_confidence_bars_overlay(
    cycles_df,
    comp_type="tSC",                 # "tSC" or "PC"
    bins=np.linspace(0, 1, 21),      # bin edges
    color_phasic="#1f77b4",          # blue
    color_tonic="#ff7f0e",           # orange
    adaptive_x=True,                 # adaptive x limits
    x_percentile=None,               # e.g., (1,99)
    x_pad=0.02                       # margin added to x-limits
):
    """
    Draws one bar chart of confidence (max prob) for both phasic and tonic,
    overlaid with transparency and normalized to show probability per bin (sum=1).
    Mean labels are automatically offset vertically to avoid overlap.
    """
    key = f"{comp_type}_conf"
    if key not in cycles_df.columns:
        raise KeyError(f"Column '{key}' not found in cycles_df.")

    def _vec(df, typ):
        v = df.loc[df["cycle_type"] == typ, key].to_numpy(dtype=float)
        return v[np.isfinite(v)]

    have_phasic = "phasic" in cycles_df["cycle_type"].unique()
    have_tonic  = "tonic"  in cycles_df["cycle_type"].unique()
    phasic = _vec(cycles_df, "phasic") if have_phasic else np.array([])
    tonic  = _vec(cycles_df, "tonic")  if have_tonic  else np.array([])

    edges = np.linspace(0, 1, bins + 1) if isinstance(bins, int) else np.asarray(bins, float)

    def _hist(v):
        if v.size == 0:
            return np.array([]), edges
        counts, _ = np.histogram(v, bins=edges)
        probs = counts / counts.sum() if counts.sum() > 0 else counts
        return probs, edges

    def _xlim_for(v_list, counts_list, edges):
        if not adaptive_x:
            return edges[0], edges[-1]
        vals = np.concatenate([v for v in v_list if v.size])
        if vals.size == 0:
            return edges[0], edges[-1]
        if x_percentile is not None:
            lo, hi = np.percentile(vals, x_percentile)
        else:
            nz = np.where(np.sum(np.vstack([c for c in counts_list if c.size]), axis=0) > 0)[0]
            if nz.size == 0:
                return edges[0], edges[-1]
            lo, hi = edges[nz[0]], edges[nz[-1] + 1]
        lo = max(0.0, lo - x_pad)
        hi = min(1.0, hi + x_pad)
        if hi - lo < 0.1:
            mid = 0.5 * (lo + hi)
            lo, hi = max(0.0, mid - 0.05), min(1.0, mid + 0.05)
        return lo, hi

    # compute histograms
    phasic_probs, edges = _hist(phasic)
    tonic_probs,  _     = _hist(tonic)
    centers = 0.5 * (edges[:-1] + edges[1:])
    widths  = np.diff(edges)

    # figure
    fig, ax = plt.subplots(figsize=(8, 4))

    # overlay both as bars with transparency
    if have_phasic:
        ax.bar(centers, phasic_probs, width=widths, align="center",
               edgecolor="black", linewidth=0.5, color=color_phasic, alpha=0.6,
               label=f"Phasic (N={phasic.size})")
    if have_tonic:
        ax.bar(centers, tonic_probs, width=widths, align="center",
               edgecolor="black", linewidth=0.5, color=color_tonic, alpha=0.6,
               label=f"Tonic (N={tonic.size})")

    # adaptive limits
    xlo, xhi = _xlim_for([phasic, tonic], [phasic_probs, tonic_probs], edges)
    ax.set_xlim(xlo, xhi)
    ymax = max(
        phasic_probs.max() if phasic_probs.size else 0,
        tonic_probs.max() if tonic_probs.size else 0,
        1e-6
    ) * 1.05
    ax.set_ylim(0, ymax)

    # labels
    ax.set_xlabel(f"{comp_type} max probability (confidence)")
    ax.set_ylabel("Probability per bin")
    ax.set_title(f"{comp_type} confidence — Phasic vs. Tonic")
    ax.legend(frameon=False)

    # mean lines and text labels with vertical offset
    offset_steps = [0.97, 0.92]  # relative y positions for phasic, tonic
    for i, (v, color, name) in enumerate([
        (phasic, color_phasic, "phasic"),
        (tonic, color_tonic, "tonic")
    ]):
        if v.size:
            mu = float(np.nanmean(v))
            if xlo <= mu <= xhi:
                ax.axvline(mu, color=color, linestyle="--", linewidth=1.2)
                ypos = ymax * offset_steps[i % len(offset_steps)]
                ax.text(mu, ypos, f"{name} mean={mu:.2f}", color=color,
                        ha="center", va="top")

    plt.tight_layout()
    plt.show()

    print(f"{comp_type} confidence — N(phasic)={phasic.size}, N(tonic)={tonic.size}")

In [ ]:
# Raw counts, adaptive x from non-empty bins (default), small padding
plot_confidence_bars_overlay(cycles_df, comp_type="tSC", bins=np.linspace(0,1,21))


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def compute_embedding_and_features(
    all_rats_data: dict,
    cycles_df: pd.DataFrame,
    mode_col: str = "mode_freq_hz_featZ_smooth",
    scale_features: bool = True,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42,
):
    """
    Build the merged dataframe (df_plot), select features (with debug prints),
    scale (optional), compute UMAP (t-SNE fallback), and return everything
    needed for plotting.
    Returns:
        emb (np.ndarray): shape (n_samples, 2)
        df_plot (pd.DataFrame): merged, filtered data aligned across sources
        color (np.ndarray): vector used for coloring (mode_col)
        xlab, ylab, method (str): axis labels and method name
        vmin, vmax (float): color scaling limits (p1, p99)
    """
    # -------- 1) Concatenate waveforms --------
    wf_frames = []
    for rid, payload in all_rats_data.items():
        wf = payload.get("waveforms_df")
        if wf is None or len(wf) == 0:
            continue
        w = wf.copy()
        if "rat_id" not in w.columns:
            w["rat_id"] = str(rid)  # ensure string
        wf_frames.append(w)

    if not wf_frames:
        raise RuntimeError("No waveforms_df found in all_rats_data.")

    waveforms_all = pd.concat(wf_frames, ignore_index=True)

    # -------- 2) Coerce key dtypes consistently --------
    def _coerce_keys(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        if "rat_id" in out.columns:
            out["rat_id"] = out["rat_id"].astype(str)
        if "cycle_type" in out.columns:
            out["cycle_type"] = out["cycle_type"].astype(str).str.lower()
        return out

    def _group_order_index(df: pd.DataFrame, group_keys):
        g = df[group_keys].copy().fillna("__NA__")
        return g.groupby(group_keys).cumcount()

    def _safe_sort(df: pd.DataFrame, prefer_order=("rat_id","cycle_type","date","SD","trial","interval_idx","cycle_idx_in_interval")):
        keys = [k for k in prefer_order if k in df.columns]
        return df.sort_values(keys, kind="mergesort") if keys else df

    waveforms_all = _coerce_keys(_safe_sort(waveforms_all))
    cycles_all    = _coerce_keys(_safe_sort(cycles_df))

    if "rat_id" not in cycles_all.columns or "cycle_type" not in cycles_all.columns:
        raise RuntimeError("cycles_df must contain 'rat_id' and 'cycle_type'.")
    if mode_col not in cycles_all.columns:
        raise RuntimeError(f"`{mode_col}` not found in cycles_df. Available: {list(cycles_all.columns)}")

    # -------- 3) Align rows via stable index within (rat_id, cycle_type) ------
    waveforms_all["gidx"] = _group_order_index(waveforms_all, ["rat_id", "cycle_type"])
    cycles_all["gidx"]    = _group_order_index(cycles_all,    ["rat_id", "cycle_type"])

    df_color = cycles_all[["rat_id","cycle_type","gidx", mode_col]].copy()
    df_plot  = waveforms_all.merge(df_color, on=["rat_id","cycle_type","gidx"], how="inner")

    if df_plot.empty:
        raise RuntimeError("No overlap between waveforms_all and cycles_df after alignment.")

    # -------- 4) Build feature matrix (INCLUDES the chosen mode_col) ----------
    numeric_cols = df_plot.select_dtypes(include=[np.number]).columns.tolist()

    meta_cols = {
        "rat_id","condition","trial","cycle_type","SD","date",
        "interval_idx","cycle_idx_in_interval","gidx"
    }
    # exclude all tSC-related numeric columns
    meta_cols |= {c for c in df_plot.columns if c.startswith("tSC")}
    # exclude all mode_* columns EXCEPT the one we want to include
    all_mode_cols = {c for c in df_plot.columns if c.startswith("mode_freq_hz_")}
    meta_cols |= (all_mode_cols - {mode_col})

    feature_cols = [c for c in numeric_cols if c not in meta_cols]

    # make sure the desired mode column is present & numeric; include it if not already
    if mode_col not in feature_cols:
        df_plot[mode_col] = pd.to_numeric(df_plot[mode_col], errors="coerce")
        feature_cols.append(mode_col)

    if not feature_cols:
        raise RuntimeError("No numeric feature columns found for embedding.")

    # ------- DEBUG PRINTS: exactly what goes into UMAP ------------------------
    print("\n=== Feature column summary ===")
    print(f"Number of features going into UMAP: {len(feature_cols)}")
    print("Feature columns:")
    for c in feature_cols:
        print("  ", c)

    print("\nPreview of the feature matrix (first 5 rows):")
    print(df_plot[feature_cols].head())

    print("\nNumeric summary (describe):")
    print(df_plot[feature_cols].describe().T)

    # -------------------------------------------------------------------------
    X = df_plot[feature_cols].to_numpy()

    from sklearn.preprocessing import StandardScaler
    if scale_features:
        X = StandardScaler().fit_transform(X)

    # color by the same mode column
    color = df_plot[mode_col].to_numpy().astype(float)

    # drop rows with non-finite entries in either X or color
    valid = np.isfinite(color) & np.isfinite(X).all(axis=1)
    df_plot = df_plot.loc[valid].reset_index(drop=True)
    X = X[valid]
    color = color[valid]

    # -------- 5) Compute embedding (UMAP→t-SNE fallback) ----------------------
    try:
        import umap
        reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                            n_components=2, random_state=random_state)
        emb = reducer.fit_transform(X)
        xlab, ylab, method = "UMAP-1", "UMAP-2", "UMAP"
    except Exception:
        from sklearn.manifold import TSNE
        tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto',
                    init='pca', random_state=random_state)
        emb = tsne.fit_transform(X)
        xlab, ylab, method = "Dim 1", "Dim 2", "t-SNE"

    # Color scaling (consistent across all plots)
    vmin = np.nanpercentile(color, 1)
    vmax = np.nanpercentile(color, 99)

    return emb, df_plot, color, xlab, ylab, method, vmin, vmax


In [ ]:
# ---- UMAP embedding colored by peak gamma ----
emb_gamma, df_plot_gamma, color_gamma, xlab_g, ylab_g, method_g, vmin_g, vmax_g = compute_embedding_and_features(
    all_rats_data=all_rats,
    cycles_df=cycles_df,
    mode_col="mode_freq_hz_featZ_smooth",
    scale_features=True,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
)

In [ ]:
# ---- Big combined figure: Rats 3,4,7,8 on top row, others on bottom row ----
# Each rat gets a column with TONIC (upper) and PHASIC (lower) subplot.

all_rats_ids = sorted(df_plot_gamma["rat_id"].unique())
_cast = type(all_rats_ids[0])
top_rats = [_cast(r) for r in [3, 4, 7, 8] if _cast(r) in all_rats_ids]
bottom_rats = [r for r in all_rats_ids if r not in top_rats]

n_top = len(top_rats)
n_bot = len(bottom_rats)
n_cols = max(n_top, n_bot)

mode_col = "mode_freq_hz_featZ_smooth"
dim_x, dim_y = 0, 1
method = method_g
xlab_dynamic = f"{method}-{dim_x + 1}"
ylab_dynamic = f"{method}-{dim_y + 1}"

if vmin_g is None or vmax_g is None:
    _vmin = np.nanpercentile(color_gamma, 1)
    _vmax = np.nanpercentile(color_gamma, 99)
else:
    _vmin, _vmax = vmin_g, vmax_g

# Use gridspec for precise control: 4 rows x (n_cols + 1) — extra col for colorbars
from matplotlib.gridspec import GridSpec

fig = plt.figure(figsize=(5.5 * n_cols + 1.2, 18))
gs = GridSpec(4, n_cols + 1, figure=fig,
              width_ratios=[1] * n_cols + [0.05],
              hspace=0.25, wspace=0.3)

axes = np.empty((4, n_cols), dtype=object)
ax0 = None
for row in range(4):
    for col in range(n_cols):
        axes[row, col] = fig.add_subplot(gs[row, col],
                                          sharex=ax0, sharey=ax0)
        if ax0 is None:
            ax0 = axes[row, col]

def _plot_rat_col(ax_tonic, ax_phasic, rid):
    m_rat = df_plot_gamma["rat_id"].values == rid
    tonic_m = m_rat & (df_plot_gamma["cycle_type"].values == "tonic")
    phasic_m = m_rat & (df_plot_gamma["cycle_type"].values == "phasic")

    sc_t = ax_tonic.scatter(
        emb_gamma[tonic_m, dim_x], emb_gamma[tonic_m, dim_y],
        c=color_gamma[tonic_m], cmap="Blues", vmin=_vmin, vmax=_vmax,
        s=14, alpha=0.8, edgecolors="none",
    )
    ax_tonic.set_title(f"Rat {rid} — TONIC", fontsize=10)

    sc_p = ax_phasic.scatter(
        emb_gamma[phasic_m, dim_x], emb_gamma[phasic_m, dim_y],
        c=color_gamma[phasic_m], cmap="Reds", vmin=_vmin, vmax=_vmax,
        s=20, marker="X", edgecolors="k", linewidths=0.2,
    )
    ax_phasic.set_title(f"Rat {rid} — PHASIC", fontsize=10)
    return sc_t, sc_p

# ---- Top rows (0-1): top_rats ----
for col_idx, rid in enumerate(top_rats):
    _plot_rat_col(axes[0, col_idx], axes[1, col_idx], rid)
for col_idx in range(n_top, n_cols):
    axes[0, col_idx].set_visible(False)
    axes[1, col_idx].set_visible(False)

# ---- Bottom rows (2-3): bottom_rats ----
for col_idx, rid in enumerate(bottom_rats):
    _plot_rat_col(axes[2, col_idx], axes[3, col_idx], rid)
for col_idx in range(n_bot, n_cols):
    axes[2, col_idx].set_visible(False)
    axes[3, col_idx].set_visible(False)

# ---- Axis labels on edges only ----
for row in range(4):
    axes[row, 0].set_ylabel(ylab_dynamic, fontsize=9)
for col_idx in range(n_cols):
    if axes[1, col_idx].get_visible():
        axes[1, col_idx].set_xlabel(xlab_dynamic, fontsize=9)
    if axes[3, col_idx].get_visible():
        axes[3, col_idx].set_xlabel(xlab_dynamic, fontsize=9)

# ---- Colorbars in the extra gridspec column ----
# Tonic colorbar spanning rows 0 & 2
cax_tonic = fig.add_subplot(gs[0:2, -1])
sm_tonic = plt.cm.ScalarMappable(cmap="Blues", norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
cb_t = fig.colorbar(sm_tonic, cax=cax_tonic)
cb_t.set_label(f"{mode_col} (Hz) — Tonic", fontsize=10)

# Phasic colorbar spanning rows 1 & 3
cax_phasic = fig.add_subplot(gs[2:4, -1])
sm_phasic = plt.cm.ScalarMappable(cmap="Reds", norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
cb_p = fig.colorbar(sm_phasic, cax=cax_phasic)
cb_p.set_label(f"{mode_col} (Hz) — Phasic", fontsize=10)

fig.suptitle(f"Peak Gamma per Theta Cycle\nTop: Positive Rats {top_rats}  |  Bottom: Control Rats {bottom_rats}",
             fontsize=13, fontweight="bold", y=0.98)
plt.show()

## Analysis — Positive vs Control

Positive rats: 3, 4, 7, 8  
Control rats: 1, 2, 6, 9

In [ ]:
# ============================================================
# Run separate PCA/ICA pipelines for Positive vs Control
# ============================================================
POSITIVE_RATS = [3, 4, 7, 8]
CONTROL_RATS  = [1, 2, 6, 9]

all_rats_positive = {k: v for k, v in all_rats.items() if k in POSITIVE_RATS}
all_rats_control  = {k: v for k, v in all_rats.items() if k in CONTROL_RATS}

print("=" * 60)
print("POSITIVE RATS:", list(all_rats_positive.keys()))
print("=" * 60)
cycles_positive, tSC_positive = run_multi_rat_tSC_pipeline(
    all_rats_positive,
    n_pca=5,
    freq_vec=np.arange(15, 141, 1),
    zscore_features=True,
    mad_k=2.0,
    weighted_alpha=0.4,
    ignore_edge_bins=1
)

print("\n" + "=" * 60)
print("CONTROL RATS:", list(all_rats_control.keys()))
print("=" * 60)
cycles_control, tSC_control = run_multi_rat_tSC_pipeline(
    all_rats_control,
    n_pca=5,
    freq_vec=np.arange(15, 141, 1),
    zscore_features=True,
    mad_k=2.0,
    weighted_alpha=0.4,
    ignore_edge_bins=1
)

In [ ]:
# ============================================================
# Cycle counts & tSC weight plots — Positive vs Control
# ============================================================
print("=== POSITIVE RATS ===")
print_cycle_counts(cycles_positive)
plot_tSC_weights(tSC_positive["weights_freq"], tSC_positive["freq_vec"])

print("\n=== CONTROL RATS ===")
print_cycle_counts(cycles_control)
plot_tSC_weights(tSC_control["weights_freq"], tSC_control["freq_vec"])

In [ ]:
# ============================================================
# Mode frequency histograms per component — Positive
# ============================================================
print("=== POSITIVE RATS ===")
plot_mode_freq_histograms_per_component(
    cycles_positive,
    col=["mode_freq_hz_featZ_smooth5_pad"],
    freq_bins=np.linspace(15, 141, 30),
    ymax=0.25,
    weights=tSC_positive["weights_freq"],
    freq_vec=tSC_positive["freq_vec"]
)

In [ ]:
# ============================================================
# Mode frequency histograms per component — Control
# ============================================================
print("=== CONTROL RATS ===")
plot_mode_freq_histograms_per_component(
    cycles_control,
    col=["mode_freq_hz_featZ_smooth5_pad"],
    freq_bins=np.linspace(15, 141, 30),
    ymax=0.25,
    weights=tSC_control["weights_freq"],
    freq_vec=tSC_control["freq_vec"]
)

In [ ]:
# ============================================================
# PCA/ICA probabilities — Positive vs Control
# ============================================================
# Recompute probabilities using each group's own PCA/ICA decomposition.

def add_prob_columns(df, results, label=""):
    """Add PCA scores + ICA/PCA probabilities to a cycles_df using its own tSC_results."""
    pca_obj    = results["pca"]
    ica_obj    = results["ica"]
    X_featZ    = results["X_cycles_featZ"]
    S          = results["strengths"]
    evr        = pca_obj.explained_variance_ratio_
    n_comp     = S.shape[1]

    assert X_featZ.shape[0] == df.shape[0], f"Row mismatch: X_featZ vs df ({label})"
    assert S.shape[0]       == df.shape[0], f"Row mismatch: S vs df ({label})"

    Z = pca_obj.transform(X_featZ)

    # --- ICA/tSC probabilities ---
    S_abs = np.abs(S)
    S_sigma = _mad_scale(S_abs, axis=0)
    S_abs_mad = S_abs / S_sigma

    tau = 1.0
    probs_tSC_plain = _softmax(S_abs,     axis=1, tau=tau)
    probs_tSC_mad   = _softmax(S_abs_mad, axis=1, tau=tau)
    probs_tSC       = probs_tSC_mad

    tSC_top_idx = np.argmax(probs_tSC, axis=1)
    tSC_conf    = probs_tSC[np.arange(probs_tSC.shape[0]), tSC_top_idx]
    tSC_entropy = -(probs_tSC * np.log(probs_tSC + 1e-12)).sum(axis=1)

    # --- PCA probabilities ---
    w_sqrt = np.sqrt(evr)[None, :]
    w_evr  = evr[None, :]
    pca_score_mag = np.abs(Z) * w_sqrt
    pca_score_pow = (Z**2)   * w_evr

    probs_PC_mag = _softmax(pca_score_mag, axis=1, tau=tau)
    probs_PC_pow = _softmax(pca_score_pow, axis=1, tau=tau)
    probs_PC     = probs_PC_mag

    PC_top_idx = np.argmax(probs_PC, axis=1)
    PC_conf    = probs_PC[np.arange(probs_PC.shape[0]), PC_top_idx]
    PC_entropy = -(probs_PC * np.log(probs_PC + 1e-12)).sum(axis=1)

    # --- Write into df ---
    for k in range(n_comp):
        df[f"PC{k+1}_score"]      = Z[:, k]
        df[f"PC{k+1}_prob"]       = probs_PC[:, k]
        df[f"PC{k+1}_prob_pow"]   = probs_PC_pow[:, k]
        df[f"tSC{k+1}_prob"]      = probs_tSC[:, k]
        df[f"tSC{k+1}_prob_plain"]= probs_tSC_plain[:, k]
        df[f"tSC{k+1}_zmad"]      = S_abs_mad[:, k]

    df["PC_top"]      = (PC_top_idx + 1).astype(int)
    df["PC_conf"]     = PC_conf
    df["PC_entropy"]  = PC_entropy
    df["tSC_top"]     = (tSC_top_idx + 1).astype(int)
    df["tSC_conf"]    = tSC_conf
    df["tSC_entropy"] = tSC_entropy

    return n_comp

# Apply to each group using its own decomposition
n_comp_pos = add_prob_columns(cycles_positive, tSC_positive, label="positive")
n_comp_ctl = add_prob_columns(cycles_control,  tSC_control,  label="control")

print("=== POSITIVE RATS ===")
plot_prob_summaries(cycles_positive, comp_type="tSC", n_comp=n_comp_pos)
plot_prob_summaries(cycles_positive, comp_type="PC",  n_comp=n_comp_pos)

print("\n=== CONTROL RATS ===")
plot_prob_summaries(cycles_control, comp_type="tSC", n_comp=n_comp_ctl)
plot_prob_summaries(cycles_control, comp_type="PC",  n_comp=n_comp_ctl)

In [ ]:
# ============================================================
# Confidence bars overlay — Positive vs Control
# ============================================================
print("=== POSITIVE RATS — tSC confidence ===")
plot_confidence_bars_overlay(cycles_positive, comp_type="tSC", bins=np.linspace(0, 1, 21))

print("\n=== CONTROL RATS — tSC confidence ===")
plot_confidence_bars_overlay(cycles_control, comp_type="tSC", bins=np.linspace(0, 1, 21))

# umap

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes

def compute_embedding_and_features(
    all_rats_data: dict,
    cycles_df: pd.DataFrame,
    mode_col: str = "mode_freq_hz_featZ_smooth",
    scale_features: bool = True,
    n_neighbors: int = 15,
    min_dist: float = 0.1,
    random_state: int = 42,
):
    """
    Build the merged dataframe (df_plot), select features (with debug prints),
    scale (optional), compute UMAP (t-SNE fallback), and return everything
    needed for plotting.
    Returns:
        emb (np.ndarray): shape (n_samples, 2)
        df_plot (pd.DataFrame): merged, filtered data aligned across sources
        color (np.ndarray): vector used for coloring (mode_col)
        xlab, ylab, method (str): axis labels and method name
        vmin, vmax (float): color scaling limits (p1, p99)
    """
    # -------- 1) Concatenate waveforms --------
    wf_frames = []
    for rid, payload in all_rats_data.items():
        wf = payload.get("waveforms_df")
        if wf is None or len(wf) == 0:
            continue
        w = wf.copy()
        if "rat_id" not in w.columns:
            w["rat_id"] = str(rid)  # ensure string
        wf_frames.append(w)

    if not wf_frames:
        raise RuntimeError("No waveforms_df found in all_rats_data.")

    waveforms_all = pd.concat(wf_frames, ignore_index=True)

    # -------- 2) Coerce key dtypes consistently --------
    def _coerce_keys(df: pd.DataFrame) -> pd.DataFrame:
        out = df.copy()
        if "rat_id" in out.columns:
            out["rat_id"] = out["rat_id"].astype(str)
        if "cycle_type" in out.columns:
            out["cycle_type"] = out["cycle_type"].astype(str).str.lower()
        return out

    def _group_order_index(df: pd.DataFrame, group_keys):
        g = df[group_keys].copy().fillna("__NA__")
        return g.groupby(group_keys).cumcount()

    def _safe_sort(df: pd.DataFrame, prefer_order=("rat_id","cycle_type","date","SD","trial","interval_idx","cycle_idx_in_interval")):
        keys = [k for k in prefer_order if k in df.columns]
        return df.sort_values(keys, kind="mergesort") if keys else df

    waveforms_all = _coerce_keys(_safe_sort(waveforms_all))
    cycles_all    = _coerce_keys(_safe_sort(cycles_df))

    if "rat_id" not in cycles_all.columns or "cycle_type" not in cycles_all.columns:
        raise RuntimeError("cycles_df must contain 'rat_id' and 'cycle_type'.")
    if mode_col not in cycles_all.columns:
        raise RuntimeError(f"`{mode_col}` not found in cycles_df. Available: {list(cycles_all.columns)}")

    # -------- 3) Align rows via stable index within (rat_id, cycle_type) ------
    waveforms_all["gidx"] = _group_order_index(waveforms_all, ["rat_id", "cycle_type"])
    cycles_all["gidx"]    = _group_order_index(cycles_all,    ["rat_id", "cycle_type"])

    df_color = cycles_all[["rat_id","cycle_type","gidx", mode_col]].copy()
    df_plot  = waveforms_all.merge(df_color, on=["rat_id","cycle_type","gidx"], how="inner")

    if df_plot.empty:
        raise RuntimeError("No overlap between waveforms_all and cycles_df after alignment.")

    # -------- 4) Build feature matrix (INCLUDES the chosen mode_col) ----------
    numeric_cols = df_plot.select_dtypes(include=[np.number]).columns.tolist()

    meta_cols = {
        "rat_id","condition","trial","cycle_type","SD","date",
        "interval_idx","cycle_idx_in_interval","gidx"
    }
    # exclude all tSC-related numeric columns
    meta_cols |= {c for c in df_plot.columns if c.startswith("tSC")}
    # exclude all mode_* columns EXCEPT the one we want to include
    all_mode_cols = {c for c in df_plot.columns if c.startswith("mode_freq_hz_")}
    meta_cols |= (all_mode_cols - {mode_col})

    feature_cols = [c for c in numeric_cols if c not in meta_cols]

    # make sure the desired mode column is present & numeric; include it if not already
    if mode_col not in feature_cols:
        df_plot[mode_col] = pd.to_numeric(df_plot[mode_col], errors="coerce")
        feature_cols.append(mode_col)

    if not feature_cols:
        raise RuntimeError("No numeric feature columns found for embedding.")

    # ------- DEBUG PRINTS: exactly what goes into UMAP ------------------------
    print("\n=== Feature column summary ===")
    print(f"Number of features going into UMAP: {len(feature_cols)}")
    print("Feature columns:")
    for c in feature_cols:
        print("  ", c)

    print("\nPreview of the feature matrix (first 5 rows):")
    print(df_plot[feature_cols].head())

    print("\nNumeric summary (describe):")
    print(df_plot[feature_cols].describe().T)

    # -------------------------------------------------------------------------
    X = df_plot[feature_cols].to_numpy()

    from sklearn.preprocessing import StandardScaler
    if scale_features:
        X = StandardScaler().fit_transform(X)

    # color by the same mode column
    color = df_plot[mode_col].to_numpy().astype(float)

    # drop rows with non-finite entries in either X or color
    valid = np.isfinite(color) & np.isfinite(X).all(axis=1)
    df_plot = df_plot.loc[valid].reset_index(drop=True)
    X = X[valid]
    color = color[valid]

    # -------- 5) Compute embedding (UMAP→t-SNE fallback) ----------------------
    try:
        import umap
        reducer = umap.UMAP(n_neighbors=n_neighbors, min_dist=min_dist,
                            n_components=2, random_state=random_state)
        emb = reducer.fit_transform(X)
        xlab, ylab, method = "UMAP-1", "UMAP-2", "UMAP"
    except Exception:
        from sklearn.manifold import TSNE
        tsne = TSNE(n_components=2, perplexity=30, learning_rate='auto',
                    init='pca', random_state=random_state)
        emb = tsne.fit_transform(X)
        xlab, ylab, method = "Dim 1", "Dim 2", "t-SNE"

    # Color scaling (consistent across all plots)
    vmin = np.nanpercentile(color, 1)
    vmax = np.nanpercentile(color, 99)

    return emb, df_plot, color, xlab, ylab, method, vmin, vmax


In [ ]:
def plot_umap_results(
    emb: np.ndarray,
    df_plot: pd.DataFrame,
    color: np.ndarray,
    xlab: str,
    ylab: str,
    method: str,
    mode_col: str = "mode_freq_hz_featZ_smooth",
    vmin: float = None,
    vmax: float = None,
    small_multiples: bool = True,
):
    """
    Plot the all-rats figure (tonic=Blues, phasic=Reds) with TWO colorbars,
    then one figure per rat, each with TWO colorbars as well.
    """
    if vmin is None or vmax is None:
        vmin = np.nanpercentile(color, 1)
        vmax = np.nanpercentile(color, 99)

    # Masks
    phasic_m_all = (df_plot["cycle_type"].values == "phasic")
    tonic_m_all  = (df_plot["cycle_type"].values == "tonic")

    # -------- All rats together --------
    fig, ax = plt.subplots(figsize=(12, 6))

    sc_tonic = ax.scatter(
        emb[tonic_m_all, 0], emb[tonic_m_all, 1],
        c=color[tonic_m_all], cmap="Blues", vmin=vmin, vmax=vmax,
        s=28, alpha=0.80, edgecolors='none', label="Tonic", zorder=2
    )
    sc_phasic = ax.scatter(
        emb[phasic_m_all, 0], emb[phasic_m_all, 1],
        c=color[phasic_m_all], cmap="Reds", vmin=vmin, vmax=vmax,
        s=42, marker='X', edgecolors='k', linewidths=0.25, label="Phasic", zorder=3
    )

    ax.set_xlabel(xlab, fontsize=11)
    ax.set_ylabel(ylab, fontsize=11)
    ax.set_title(f"{method} of waveform features (+ {mode_col}) — color: {mode_col} (Hz)", fontsize=12)
    ax.legend(frameon=False, loc='upper left', fontsize=10)

    # Get the position of the main axes
    pos = ax.get_position()
    
    # Create two colorbar axes with same vertical positioning
    cbar_width = 0.015
    cbar_height = 0.4
    cbar_bottom = pos.y0 + (pos.height - cbar_height) / 2  # Center vertically
    
    # Tonic colorbar (left one) - no tick labels
    cax_tonic = fig.add_axes([pos.x1 + 0.02, cbar_bottom, cbar_width, cbar_height])
    cbar_tonic = fig.colorbar(sc_tonic, cax=cax_tonic)
    cbar_tonic.set_label("Tonic (Hz)", fontsize=10)
    cbar_tonic.set_ticks([])  # Remove tick labels
    
    # Phasic colorbar (right one) - with tick labels, moved further right
    cax_phasic = fig.add_axes([pos.x1 + 0.08, cbar_bottom, cbar_width, cbar_height])
    cbar_phasic = fig.colorbar(sc_phasic, cax=cax_phasic)
    cbar_phasic.set_label("Phasic (Hz)", fontsize=10)

    plt.show()

    # -------- Per-rat figures (each with two colorbars) --------
    if small_multiples:
        rats = sorted(df_plot["rat_id"].unique())
        for rid in rats:
            m_rat = (df_plot["rat_id"].values == rid)
            phasic_m = m_rat & (df_plot["cycle_type"].values == "phasic")
            tonic_m  = m_rat & (df_plot["cycle_type"].values == "tonic")

            fig, ax = plt.subplots(figsize=(8, 5.5))

            sc_tonic = ax.scatter(
                emb[tonic_m, 0], emb[tonic_m, 1],
                c=color[tonic_m], cmap="Blues", vmin=vmin, vmax=vmax,
                s=28, alpha=0.80, edgecolors='none', label="Tonic", zorder=2
            )
            sc_phasic = ax.scatter(
                emb[phasic_m, 0], emb[phasic_m, 1],
                c=color[phasic_m], cmap="Reds", vmin=vmin, vmax=vmax,
                s=42, marker='X', edgecolors='k', linewidths=0.25, label="Phasic", zorder=3
            )

            ax.set_title(f"Rat {rid} — {method} (features include {mode_col})", fontsize=12)
            ax.set_xlabel(xlab, fontsize=11)
            ax.set_ylabel(ylab, fontsize=11)
            ax.legend(frameon=False, loc='upper left', fontsize=10)

            # Get the position of the main axes
            pos = ax.get_position()
            
            # Create two colorbar axes with same vertical positioning
            cbar_width = 0.02
            cbar_height = 0.4
            cbar_bottom = pos.y0 + (pos.height - cbar_height) / 2  # Center vertically
            
            # Tonic colorbar (left one) - no tick labels
            cax_tonic = fig.add_axes([pos.x1 + 0.02, cbar_bottom, cbar_width, cbar_height])
            cbar_tonic = fig.colorbar(sc_tonic, cax=cax_tonic)
            cbar_tonic.set_label("Tonic (Hz)", fontsize=10)
            cbar_tonic.set_ticks([])  # Remove tick labels
            
            # Phasic colorbar (right one) - with tick labels, moved further right
            cax_phasic = fig.add_axes([pos.x1 + 0.09, cbar_bottom, cbar_width, cbar_height])
            cbar_phasic = fig.colorbar(sc_phasic, cax=cax_phasic)
            cbar_phasic.set_label("Phasic (Hz)", fontsize=10)

            plt.show()

In [ ]:
emb, df_plot, color, xlab, ylab, method, vmin, vmax = compute_embedding_and_features(
    all_rats_data=all_rats,
    cycles_df=cycles_df,
    mode_col="mode_freq_hz_featZ_smooth",
    scale_features=True,
    n_neighbors=15,
    min_dist=0.1,
    random_state=42,
)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

def plot_umap_results_split(
    emb: np.ndarray,
    df_plot: pd.DataFrame,
    color: np.ndarray,
    xlab: str,
    ylab: str,
    method: str,
    mode_col: str = "mode_freq_hz_featZ_smooth",
    vmin: float = None,
    vmax: float = None,
    small_multiples: bool = True,
    dim_x: int = 0,
    dim_y: int = 1,
):
    """
    Make split plots (TONIC and PHASIC in separate vertical subplots).
    - First: all rats -> 2 stacked subplots
    - Then (optional): per-rat -> one figure per rat, each with 2 stacked subplots
    You can choose which embedding dimensions to plot using dim_x and dim_y.
    """
    if vmin is None or vmax is None:
        vmin = np.nanpercentile(color, 1)
        vmax = np.nanpercentile(color, 99)

    # Sanity check
    n_dims = emb.shape[1]
    if dim_x >= n_dims or dim_y >= n_dims:
        raise ValueError(f"Embedding only has {n_dims} dimensions, but you asked for ({dim_x}, {dim_y})")

    # Update axis labels dynamically
    xlab_dynamic = f"{method}-{dim_x + 1}"
    ylab_dynamic = f"{method}-{dim_y + 1}"

    tonic_all = (df_plot["cycle_type"].values == "tonic")
    phasic_all = (df_plot["cycle_type"].values == "phasic")

    # -------------------- ALL RATS: split subplots --------------------
    fig, axes = plt.subplots(
        2, 1, sharex=True, sharey=True,
        figsize=(12, 10), constrained_layout=True
    )

    # TONIC (top)
    sc_tonic = axes[0].scatter(
        emb[tonic_all, dim_x], emb[tonic_all, dim_y],
        c=color[tonic_all], cmap="Blues", vmin=vmin, vmax=vmax,
        s=28, alpha=0.85, edgecolors='none'
    )
    axes[0].set_title(f"TONIC — {method} (+ {mode_col})", fontsize=12)
    axes[0].set_ylabel(ylab_dynamic, fontsize=11)

    # PHASIC (bottom)
    sc_phasic = axes[1].scatter(
        emb[phasic_all, dim_x], emb[phasic_all, dim_y],
        c=color[phasic_all], cmap="Reds", vmin=vmin, vmax=vmax,
        s=42, marker='X', edgecolors='k', linewidths=0.25
    )
    axes[1].set_title(f"PHASIC — {method} (+ {mode_col})", fontsize=12)
    axes[1].set_xlabel(xlab_dynamic, fontsize=11)
    axes[1].set_ylabel(ylab_dynamic, fontsize=11)

    # Colorbars for each subplot
    cbar_t = fig.colorbar(sc_tonic, ax=axes[0], fraction=0.046, pad=0.04)
    cbar_p = fig.colorbar(sc_phasic, ax=axes[1], fraction=0.046, pad=0.04)
    cbar_t.set_label(f"{mode_col} (Hz) — Tonic", fontsize=10)
    cbar_p.set_label(f"{mode_col} (Hz) — Phasic", fontsize=10)

    plt.show()

    # -------------------- PER-RAT: split subplots --------------------
    if small_multiples:
        rats = sorted(df_plot["rat_id"].unique())
        for rid in rats:
            m_rat = (df_plot["rat_id"].values == rid)
            tonic_m = m_rat & (df_plot["cycle_type"].values == "tonic")
            phasic_m = m_rat & (df_plot["cycle_type"].values == "phasic")

            fig, axes = plt.subplots(
                2, 1, sharex=True, sharey=True,
                figsize=(10, 8), constrained_layout=True
            )

            # TONIC (top)
            sc_tonic = axes[0].scatter(
                emb[tonic_m, dim_x], emb[tonic_m, dim_y],
                c=color[tonic_m], cmap="Blues", vmin=vmin, vmax=vmax,
                s=28, alpha=0.85, edgecolors='none'
            )
            axes[0].set_title(f"Rat {rid} — TONIC — {method} (+ {mode_col})", fontsize=12)
            axes[0].set_ylabel(ylab_dynamic, fontsize=11)

            # PHASIC (bottom)
            sc_phasic = axes[1].scatter(
                emb[phasic_m, dim_x], emb[phasic_m, dim_y],
                c=color[phasic_m], cmap="Reds", vmin=vmin, vmax=vmax,
                s=42, marker='X', edgecolors='k', linewidths=0.25
            )
            axes[1].set_title(f"Rat {rid} — PHASIC — {method} (+ {mode_col})", fontsize=12)
            axes[1].set_xlabel(xlab_dynamic, fontsize=11)
            axes[1].set_ylabel(ylab_dynamic, fontsize=11)

            # Colorbars for each subplot (same vmin/vmax)
            cbar_t = fig.colorbar(sc_tonic, ax=axes[0], fraction=0.046, pad=0.04)
            cbar_p = fig.colorbar(sc_phasic, ax=axes[1], fraction=0.046, pad=0.04)
            cbar_t.set_label(f"{mode_col} (Hz) — Tonic", fontsize=10)
            cbar_p.set_label(f"{mode_col} (Hz) — Phasic", fontsize=10)

            plt.show()

In [ ]:
plot_umap_results_split(
    emb, df_plot, color, xlab, ylab, method,
    mode_col="mode_freq_hz_featZ_smooth",
    vmin=vmin, vmax=vmax,
    small_multiples=True,
)

## UMAP — Positive vs Control (separate embeddings)

In [ ]:
# ============================================================
# Compute UMAP separately for Positive and Control groups
# ============================================================
POSITIVE_RATS = [3, 4, 7, 8]
CONTROL_RATS  = [1, 2, 6, 9]

mode_col = "mode_freq_hz_featZ_smooth"

print("Computing UMAP for POSITIVE rats...")
emb_pos, df_pos, color_pos, xlab_pos, ylab_pos, method_pos, vmin_pos, vmax_pos = \
    compute_embedding_and_features(
        all_rats_data=all_rats_positive,
        cycles_df=cycles_positive,
        mode_col=mode_col,
        scale_features=True,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42,
    )

print("\nComputing UMAP for CONTROL rats...")
emb_ctl, df_ctl, color_ctl, xlab_ctl, ylab_ctl, method_ctl, vmin_ctl, vmax_ctl = \
    compute_embedding_and_features(
        all_rats_data=all_rats_control,
        cycles_df=cycles_control,
        mode_col=mode_col,
        scale_features=True,
        n_neighbors=15,
        min_dist=0.1,
        random_state=42,
    )

In [ ]:
# ============================================================
# Big combined figure: Positive rats top rows, Control bottom rows
# Each rat = one column with TONIC (upper) and PHASIC (lower) subplot
# Separate UMAP embeddings per group, colored by mode_freq_hz_featZ_smooth
# ============================================================
from matplotlib.gridspec import GridSpec

mode_col = "mode_freq_hz_featZ_smooth"

# --- Identify rat IDs present in each group ---
pos_ids = sorted(df_pos["rat_id"].unique())
ctl_ids = sorted(df_ctl["rat_id"].unique())
n_pos = len(pos_ids)
n_ctl = len(ctl_ids)
n_cols = max(n_pos, n_ctl)

# --- Shared color limits across both groups ---
_vmin = min(vmin_pos, vmin_ctl)
_vmax = max(vmax_pos, vmax_ctl)

# --- Axis labels ---
dim_x, dim_y = 0, 1
xlab_pos_dyn = f"{method_pos}-{dim_x + 1}"
ylab_pos_dyn = f"{method_pos}-{dim_y + 1}"
xlab_ctl_dyn = f"{method_ctl}-{dim_x + 1}"
ylab_ctl_dyn = f"{method_ctl}-{dim_y + 1}"

# --- Figure: 4 rows x (n_cols + 1) with extra col for colorbars ---
fig = plt.figure(figsize=(5.5 * n_cols + 1.2, 18))
gs = GridSpec(4, n_cols + 1, figure=fig,
              width_ratios=[1] * n_cols + [0.05],
              hspace=0.25, wspace=0.3)

axes = np.empty((4, n_cols), dtype=object)
for row in range(4):
    for col in range(n_cols):
        axes[row, col] = fig.add_subplot(gs[row, col])


def _plot_rat_col(ax_tonic, ax_phasic, rid, emb_grp, df_grp, color_grp):
    """Scatter tonic/phasic for one rat using its group's embedding."""
    m_rat     = df_grp["rat_id"].values == rid
    tonic_m   = m_rat & (df_grp["cycle_type"].values == "tonic")
    phasic_m  = m_rat & (df_grp["cycle_type"].values == "phasic")

    ax_tonic.scatter(
        emb_grp[tonic_m, dim_x], emb_grp[tonic_m, dim_y],
        c=color_grp[tonic_m], cmap="Blues",
        vmin=_vmin, vmax=_vmax,
        s=14, alpha=0.8, edgecolors="none",
    )
    ax_tonic.set_title(f"Rat {rid} — TONIC", fontsize=10)

    ax_phasic.scatter(
        emb_grp[phasic_m, dim_x], emb_grp[phasic_m, dim_y],
        c=color_grp[phasic_m], cmap="Reds",
        vmin=_vmin, vmax=_vmax,
        s=20, marker="X", edgecolors="k", linewidths=0.2,
    )
    ax_phasic.set_title(f"Rat {rid} — PHASIC", fontsize=10)


# ---- Top rows (0-1): Positive rats (own UMAP) ----
for col_idx, rid in enumerate(pos_ids):
    _plot_rat_col(axes[0, col_idx], axes[1, col_idx],
                  rid, emb_pos, df_pos, color_pos)
for col_idx in range(n_pos, n_cols):
    axes[0, col_idx].set_visible(False)
    axes[1, col_idx].set_visible(False)

# ---- Bottom rows (2-3): Control rats (own UMAP) ----
for col_idx, rid in enumerate(ctl_ids):
    _plot_rat_col(axes[2, col_idx], axes[3, col_idx],
                  rid, emb_ctl, df_ctl, color_ctl)
for col_idx in range(n_ctl, n_cols):
    axes[2, col_idx].set_visible(False)
    axes[3, col_idx].set_visible(False)

# ---- Axis labels on edges only ----
for col_idx in range(n_cols):
    if axes[0, col_idx].get_visible():
        axes[0, col_idx].set_ylabel(ylab_pos_dyn, fontsize=9)
    if axes[2, col_idx].get_visible():
        axes[2, col_idx].set_ylabel(ylab_ctl_dyn, fontsize=9)
    if axes[1, col_idx].get_visible():
        axes[1, col_idx].set_xlabel(xlab_pos_dyn, fontsize=9)
    if axes[3, col_idx].get_visible():
        axes[3, col_idx].set_xlabel(xlab_ctl_dyn, fontsize=9)
for row in range(4):
    for col_idx in range(1, n_cols):
        if axes[row, col_idx].get_visible():
            axes[row, col_idx].set_ylabel("")

# ---- Colorbars ----
cax_tonic = fig.add_subplot(gs[0:2, -1])
sm_tonic = plt.cm.ScalarMappable(cmap="Blues",
                                  norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
cb_t = fig.colorbar(sm_tonic, cax=cax_tonic)
cb_t.set_label(f"{mode_col} (Hz) — Tonic", fontsize=10)

cax_phasic = fig.add_subplot(gs[2:4, -1])
sm_phasic = plt.cm.ScalarMappable(cmap="Reds",
                                   norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
cb_p = fig.colorbar(sm_phasic, cax=cax_phasic)
cb_p.set_label(f"{mode_col} (Hz) — Phasic", fontsize=10)

fig.suptitle(
    f"UMAP — {mode_col}\n"
    f"Top: Positive Rats {pos_ids} | Bottom: Control Rats {ctl_ids}\n"
    f"(separate PCA/ICA & UMAP per group)",
    fontsize=13, fontweight="bold", y=0.98,
)
plt.show()

In [ ]:
# ============================================================
# plot_umap_results_split — Positive group
# ============================================================
print("=== POSITIVE RATS ===")
plot_umap_results_split(
    emb_pos, df_pos, color_pos, xlab_pos, ylab_pos, method_pos,
    mode_col="mode_freq_hz_featZ_smooth",
    vmin=_vmin, vmax=_vmax,
    small_multiples=True,
)

In [ ]:
# ============================================================
# Per-rat grid figure — POSITIVE group only
# Each positive rat = one column, TONIC row + PHASIC row
# ============================================================
from matplotlib.gridspec import GridSpec

mode_col = "mode_freq_hz_featZ_smooth"
dim_x, dim_y = 0, 1

pos_ids = sorted(df_pos["rat_id"].unique())
n_pos = len(pos_ids)

xlab_dyn = f"{method_pos}-{dim_x + 1}"
ylab_dyn = f"{method_pos}-{dim_y + 1}"

fig = plt.figure(figsize=(5.5 * n_pos + 1.2, 9))
gs = GridSpec(2, n_pos + 1, figure=fig,
              width_ratios=[1] * n_pos + [0.05],
              hspace=0.25, wspace=0.3)

axes = np.empty((2, n_pos), dtype=object)
for row in range(2):
    for col in range(n_pos):
        axes[row, col] = fig.add_subplot(gs[row, col])

for col_idx, rid in enumerate(pos_ids):
    m_rat    = df_pos["rat_id"].values == rid
    tonic_m  = m_rat & (df_pos["cycle_type"].values == "tonic")
    phasic_m = m_rat & (df_pos["cycle_type"].values == "phasic")

    axes[0, col_idx].scatter(
        emb_pos[tonic_m, dim_x], emb_pos[tonic_m, dim_y],
        c=color_pos[tonic_m], cmap="Blues",
        vmin=_vmin, vmax=_vmax,
        s=14, alpha=0.8, edgecolors="none",
    )
    axes[0, col_idx].set_title(f"Rat {rid} — TONIC", fontsize=10)

    axes[1, col_idx].scatter(
        emb_pos[phasic_m, dim_x], emb_pos[phasic_m, dim_y],
        c=color_pos[phasic_m], cmap="Reds",
        vmin=_vmin, vmax=_vmax,
        s=20, marker="X", edgecolors="k", linewidths=0.2,
    )
    axes[1, col_idx].set_title(f"Rat {rid} — PHASIC", fontsize=10)

# Axis labels
for col_idx in range(n_pos):
    axes[1, col_idx].set_xlabel(xlab_dyn, fontsize=9)
axes[0, 0].set_ylabel(ylab_dyn, fontsize=9)
axes[1, 0].set_ylabel(ylab_dyn, fontsize=9)
for row in range(2):
    for col_idx in range(1, n_pos):
        axes[row, col_idx].set_ylabel("")

# Colorbars
cax_t = fig.add_subplot(gs[0, -1])
sm_t = plt.cm.ScalarMappable(cmap="Blues", norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
fig.colorbar(sm_t, cax=cax_t).set_label(f"{mode_col} (Hz) — Tonic", fontsize=10)

cax_p = fig.add_subplot(gs[1, -1])
sm_p = plt.cm.ScalarMappable(cmap="Reds", norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
fig.colorbar(sm_p, cax=cax_p).set_label(f"{mode_col} (Hz) — Phasic", fontsize=10)

fig.suptitle(
    f"UMAP — {mode_col} — Positive Rats {pos_ids}\n(separate PCA/ICA & UMAP)",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# plot_umap_results_split — Control group
# ============================================================
print("=== CONTROL RATS ===")
plot_umap_results_split(
    emb_ctl, df_ctl, color_ctl, xlab_ctl, ylab_ctl, method_ctl,
    mode_col="mode_freq_hz_featZ_smooth",
    vmin=_vmin, vmax=_vmax,
    small_multiples=True,
)

In [ ]:
# ============================================================
# Per-rat grid figure — CONTROL group only
# Each control rat = one column, TONIC row + PHASIC row
# ============================================================
from matplotlib.gridspec import GridSpec

mode_col = "mode_freq_hz_featZ_smooth"
dim_x, dim_y = 0, 1

ctl_ids = sorted(df_ctl["rat_id"].unique())
n_ctl = len(ctl_ids)

xlab_dyn = f"{method_ctl}-{dim_x + 1}"
ylab_dyn = f"{method_ctl}-{dim_y + 1}"

fig = plt.figure(figsize=(5.5 * n_ctl + 1.2, 9))
gs = GridSpec(2, n_ctl + 1, figure=fig,
              width_ratios=[1] * n_ctl + [0.05],
              hspace=0.25, wspace=0.3)

axes = np.empty((2, n_ctl), dtype=object)
for row in range(2):
    for col in range(n_ctl):
        axes[row, col] = fig.add_subplot(gs[row, col])

for col_idx, rid in enumerate(ctl_ids):
    m_rat    = df_ctl["rat_id"].values == rid
    tonic_m  = m_rat & (df_ctl["cycle_type"].values == "tonic")
    phasic_m = m_rat & (df_ctl["cycle_type"].values == "phasic")

    axes[0, col_idx].scatter(
        emb_ctl[tonic_m, dim_x], emb_ctl[tonic_m, dim_y],
        c=color_ctl[tonic_m], cmap="Blues",
        vmin=_vmin, vmax=_vmax,
        s=14, alpha=0.8, edgecolors="none",
    )
    axes[0, col_idx].set_title(f"Rat {rid} — TONIC", fontsize=10)

    axes[1, col_idx].scatter(
        emb_ctl[phasic_m, dim_x], emb_ctl[phasic_m, dim_y],
        c=color_ctl[phasic_m], cmap="Reds",
        vmin=_vmin, vmax=_vmax,
        s=20, marker="X", edgecolors="k", linewidths=0.2,
    )
    axes[1, col_idx].set_title(f"Rat {rid} — PHASIC", fontsize=10)

# Axis labels
for col_idx in range(n_ctl):
    axes[1, col_idx].set_xlabel(xlab_dyn, fontsize=9)
axes[0, 0].set_ylabel(ylab_dyn, fontsize=9)
axes[1, 0].set_ylabel(ylab_dyn, fontsize=9)
for row in range(2):
    for col_idx in range(1, n_ctl):
        axes[row, col_idx].set_ylabel("")

# Colorbars
cax_t = fig.add_subplot(gs[0, -1])
sm_t = plt.cm.ScalarMappable(cmap="Blues", norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
fig.colorbar(sm_t, cax=cax_t).set_label(f"{mode_col} (Hz) — Tonic", fontsize=10)

cax_p = fig.add_subplot(gs[1, -1])
sm_p = plt.cm.ScalarMappable(cmap="Reds", norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
fig.colorbar(sm_p, cax=cax_p).set_label(f"{mode_col} (Hz) — Phasic", fontsize=10)

fig.suptitle(
    f"UMAP — {mode_col} — Control Rats {ctl_ids}\n(separate PCA/ICA & UMAP)",
    fontsize=13, fontweight="bold", y=1.01,
)
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# UMAP — All Positive vs All Control (NO tonic/phasic split)
# One subplot per group, all cycles pooled together
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

dim_x, dim_y = 0, 1
_vmin = min(vmin_pos, vmin_ctl)
_vmax = max(vmax_pos, vmax_ctl)

fig, axes = plt.subplots(1, 2, figsize=(20, 8), constrained_layout=True)

# — Positive (all cycles) —
sc_pos = axes[0].scatter(
    emb_pos[:, dim_x], emb_pos[:, dim_y],
    c=color_pos, cmap="coolwarm", vmin=_vmin, vmax=_vmax,
    s=18, alpha=0.7, edgecolors="none",
)
axes[0].set_title("POSITIVE rats", fontsize=13)
axes[0].set_xlabel(f"{method_pos}-{dim_x+1}", fontsize=11)
axes[0].set_ylabel(f"{method_pos}-{dim_y+1}", fontsize=11)
plt.colorbar(sc_pos, ax=axes[0], label=mode_col, shrink=0.8)

# — Control (all cycles) —
sc_ctl = axes[1].scatter(
    emb_ctl[:, dim_x], emb_ctl[:, dim_y],
    c=color_ctl, cmap="coolwarm", vmin=_vmin, vmax=_vmax,
    s=18, alpha=0.7, edgecolors="none",
)
axes[1].set_title("CONTROL rats", fontsize=13)
axes[1].set_xlabel(f"{method_ctl}-{dim_x+1}", fontsize=11)
axes[1].set_ylabel(f"{method_ctl}-{dim_y+1}", fontsize=11)
plt.colorbar(sc_ctl, ax=axes[1], label=mode_col, shrink=0.8)

fig.suptitle("UMAP — Positive vs Control", fontsize=15, y=1.02)
plt.show()

In [ ]:
# ============================================================
# UMAP — All Positive vs All Control (TONIC / PHASIC split)
# 2x2 grid: rows = tonic/phasic, cols = positive/control
# ============================================================
import matplotlib.pyplot as plt
import numpy as np

dim_x, dim_y = 0, 1
_vmin = min(vmin_pos, vmin_ctl)
_vmax = max(vmax_pos, vmax_ctl)

tonic_pos  = df_pos["cycle_type"].values == "tonic"
phasic_pos = df_pos["cycle_type"].values == "phasic"
tonic_ctl  = df_ctl["cycle_type"].values == "tonic"
phasic_ctl = df_ctl["cycle_type"].values == "phasic"

fig, axes = plt.subplots(2, 2, figsize=(20, 16), constrained_layout=True)

# Row 0: TONIC
axes[0, 0].scatter(
    emb_pos[tonic_pos, dim_x], emb_pos[tonic_pos, dim_y],
    c=color_pos[tonic_pos], cmap="Blues", vmin=_vmin, vmax=_vmax,
    s=18, alpha=0.7, edgecolors="none",
)
axes[0, 0].set_title("POSITIVE — TONIC", fontsize=13)

axes[0, 1].scatter(
    emb_ctl[tonic_ctl, dim_x], emb_ctl[tonic_ctl, dim_y],
    c=color_ctl[tonic_ctl], cmap="Blues", vmin=_vmin, vmax=_vmax,
    s=18, alpha=0.7, edgecolors="none",
)
axes[0, 1].set_title("CONTROL — TONIC", fontsize=13)

# Row 1: PHASIC
axes[1, 0].scatter(
    emb_pos[phasic_pos, dim_x], emb_pos[phasic_pos, dim_y],
    c=color_pos[phasic_pos], cmap="Reds", vmin=_vmin, vmax=_vmax,
    s=24, marker="X", edgecolors="k", linewidths=0.2,
)
axes[1, 0].set_title("POSITIVE — PHASIC", fontsize=13)

axes[1, 1].scatter(
    emb_ctl[phasic_ctl, dim_x], emb_ctl[phasic_ctl, dim_y],
    c=color_ctl[phasic_ctl], cmap="Reds", vmin=_vmin, vmax=_vmax,
    s=24, marker="X", edgecolors="k", linewidths=0.2,
)
axes[1, 1].set_title("CONTROL — PHASIC", fontsize=13)

# Axis labels
for row in range(2):
    for col in range(2):
        method_lbl = method_pos if col == 0 else method_ctl
        axes[row, col].set_xlabel(f"{method_lbl}-{dim_x+1}", fontsize=11)
        axes[row, col].set_ylabel(f"{method_lbl}-{dim_y+1}", fontsize=11)

# Colorbars
sm_t = plt.cm.ScalarMappable(cmap="Blues", norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
sm_t.set_array([])
fig.colorbar(sm_t, ax=axes[0, :].tolist(), label=mode_col, shrink=0.8)

sm_p = plt.cm.ScalarMappable(cmap="Reds", norm=plt.Normalize(vmin=_vmin, vmax=_vmax))
sm_p.set_array([])
fig.colorbar(sm_p, ax=axes[1, :].tolist(), label=mode_col, shrink=0.8)

fig.suptitle("UMAP — Positive vs Control", fontsize=15, y=1.02)
plt.show()